# Qwen3-0.6B-Base、1層の計量つき TT と行列低ランク

対象は `model.model.layers[14].mlp.up_proj`（1024 入力、3072 出力、bias なし）。
セル2の `RUN_FULL_EXPERIMENT` が False のときは、検算、事前計測、動作確認まで行う。
True のときは、検算のあと 4.7 と 5.4 を飛ばし、5.1 から 5.3 の本実行まで進む。


## 4.1 設定

- 入力: プロジェクトルート、`data/hf_cache/download_record.json`
- このステップで行うこと: seed、λ、停止条件、予算、しきい値、パス、ライブラリの版、モデルとデータの revision を `settings.json` に保存する。新規の pytest を実行する。
- 行わないこと: 記事の復元、モデルの順伝播、圧縮、判定
- 出力: `settings.json`。変数 `RUN_FULL_EXPERIMENT` は False。`RECORD`、`RESULTS_DIR`、`EXPERIMENT_DATA`、`BUDGETS`、`LAMBDA`
- 完了条件: 行列 rank が予算 50% で 384、25% で 192。新規テストが成功する。CUDA が使える。
- 次への接続: 4.2 は `RECORD` と `EXPERIMENT_DATA` を使う。

In [1]:
import gc
import json
import math
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

PROJECT_ROOT = Path(r"D:\dev\nn-compression-svd-dmrg")
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / "data" / "hf_cache"))
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import torch

warnings.filterwarnings("ignore", message=r".*does not have a deterministic implementation.*")
torch.set_num_threads(8)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

from nn_compression.compression.metric_low_rank import (
    channel_rms,
    gram_diagnostics,
    gram_matrices_from_documents,
    gram_matrix,
    max_matrix_rank_within_budget,
    matrix_factor_parameter_count,
    regularize_gram,
    squared_output_error_from_activations,
    squared_output_error_from_gram,
    squared_output_errors_from_documents,
    truncated_svd_weight,
    whitened_truncated_svd,
)
from nn_compression.llm.activation_capture import collect_sequence_inputs
from nn_compression.llm.hf_loading import (
    load_causal_lm_and_tokenizer,
    load_download_record,
    load_tokenizer,
    load_wikitext,
)
from nn_compression.llm.perplexity import sliding_window_perplexity, temporary_linear_weight
from nn_compression.llm.text_documents import hash_overlaps, read_c4_texts, restore_wikitext_articles, select_token_prefix_sequences
from nn_compression.metrics.bootstrap import bootstrap_ratio_difference_intervals
from nn_compression.metrics.macs import compressed_linear_macs, linear_macs, tt_linear_direct_contraction_macs
from nn_compression.metrics.pilot_decision import classify_pilot_outcome
from nn_compression.tt_mps import (
    canonicalize_tt_matrix_cores,
    dense_to_tt_matrix_cores,
    tt_matrix_canonicality_errors,
    tt_matrix_num_parameters_from_ranks,
    tt_matrix_ranks,
    tt_matrix_to_dense,
)
from nn_compression.tt_mps.metric_tt import (
    diagonal_rms_scales,
    fit_diagonal_metric_tt,
    metric_tt_hvp,
    metric_tt_jacobi_diagonal,
    prepare_m5_from_diagonal_tt,
    q_lambda_of_cores,
    run_metric_tt_als,
    scale_matrix_columns,
    select_balanced_tt_matrix_modes,
    select_uniform_tt_rank,
    tt_parameter_count,
    uniform_tt_matrix_ranks,
    update_tt_matrix_site,
)
from nn_compression.utils import atomic_torch_save, atomic_write_csv, atomic_write_json, get_experiment_dirs, set_seed

RUN_FULL_EXPERIMENT = True
SEED = 0
LAMBDA = 1e-6
CG_RELATIVE_TOLERANCE = 1e-8
CG_MAX_ITERATIONS = 500
SWEEP_RELATIVE_DECREASE = 1e-6
SWEEP_MAX = 50
SEQUENCE_LENGTH = 1024
MIN_TOKENS = 1024
CALIBRATION_LIMIT = 128
SENSITIVITY_LIMIT = 16
VALIDATION_LIMIT = 32
BOOTSTRAP_RESAMPLES = 1000
RHO_LOW = 0.2
RHO_HIGH = 0.5
MIN_RELATIVE_G = 0.05
LAYER_INDEX = 14
OUT_FEATURES = 3072
IN_FEATURES = 1024
BUDGETS = {"50": 1_572_864, "25": 786_432}
EXPECTED_MATRIX_RANKS = {"50": 384, "25": 192}

set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA が使えません。")

DATA_DIR, MODELS_DIR, RESULTS_DIR = get_experiment_dirs(
    PROJECT_ROOT,
    "50_llm_tt_vs_matrix",
    "10_qwen3_0p6b_base",
    "01_single_layer_pilot",
)
EXPERIMENT_DATA = DATA_DIR / "50_llm_tt_vs_matrix" / "10_qwen3_0p6b_base" / "01_single_layer_pilot"
EXPERIMENT_DATA.mkdir(parents=True, exist_ok=True)
RECORD = load_download_record(PROJECT_ROOT / "data" / "hf_cache" / "download_record.json")

import datasets
import transformers

for label, budget in BUDGETS.items():
    rank = max_matrix_rank_within_budget(OUT_FEATURES, IN_FEATURES, budget)
    if rank != EXPECTED_MATRIX_RANKS[label]:
        raise RuntimeError(f"予算 {label} の行列 rank が {rank} です。{EXPECTED_MATRIX_RANKS[label]} である必要があります。")
    parameters = matrix_factor_parameter_count(OUT_FEATURES, IN_FEATURES, rank)
    if parameters > budget:
        raise RuntimeError(f"予算 {label} の行列パラメータ数 {parameters} が上限を超えています。")

new_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_qwen3_metric_pilot_checks.py", "-q", "--tb=line"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    encoding="utf-8",
    errors="replace",
)
print(new_tests.stdout)
print(new_tests.stderr)
if new_tests.returncode != 0:
    raise RuntimeError("新規の pytest が失敗しました。")

internal_test = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_core_api_v1_docs_sync.py::test_visible_internal_helpers_are_listed_in_internal_boundary",
        "-q",
        "--tb=line",
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    encoding="utf-8",
    errors="replace",
)
internal_log = (internal_test.stdout or "") + "\n" + (internal_test.stderr or "")
print(internal_log)
if "`gram_matrix`" in internal_log or "gram_matrixの定義" in internal_log:
    raise RuntimeError("gram_matrix が Internal_API に見つかりません。")

PYTEST_LOG = {
    "new_tests_returncode": new_tests.returncode,
    "new_tests_stdout": new_tests.stdout,
    "new_tests_stderr": new_tests.stderr,
    "internal_helper_returncode": internal_test.returncode,
    "internal_helper_log": internal_log,
    "full_suite_not_rerun": {
        "reason": "Notebook 11 は再描画フラグが真のまま実行され、既存の結果を書き換え得る。全スイートはこの Notebook では再実行しない。",
        "earlier_run_in_this_task": "5 failed, 2385 passed, 1 skipped, 3 errors, 35.27s",
    },
}

def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float):
        if math.isnan(value):
            return "nan"
        if math.isinf(value):
            return "inf" if value > 0 else "-inf"
        return value
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return json_ready(float(value))
    return value

SETTINGS = {
    "run_full_experiment": RUN_FULL_EXPERIMENT,
    "seed": SEED,
    "lambda": LAMBDA,
    "cg_relative_tolerance": CG_RELATIVE_TOLERANCE,
    "cg_max_iterations": CG_MAX_ITERATIONS,
    "sweep_relative_decrease": SWEEP_RELATIVE_DECREASE,
    "sweep_max": SWEEP_MAX,
    "sequence_length": SEQUENCE_LENGTH,
    "min_tokens": MIN_TOKENS,
    "calibration_sequences": CALIBRATION_LIMIT,
    "sensitivity_sequences": SENSITIVITY_LIMIT,
    "validation_sequences": VALIDATION_LIMIT,
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    "bootstrap_seed": SEED,
    "rho_low": RHO_LOW,
    "rho_high": RHO_HIGH,
    "min_relative_g": MIN_RELATIVE_G,
    "budgets": BUDGETS,
    "matrix_ranks": EXPECTED_MATRIX_RANKS,
    "layer": "model.model.layers[14].mlp.up_proj",
    "gate_layer": "model.model.layers[14].mlp.gate_proj",
    "feature_permutation": False,
    "optimization_dtype": "float64",
    "optimization_device": "cpu",
    "forward_dtype": "float32",
    "perplexity": {
        "text": "WikiText-2 validation の text 列を改行2つで連結",
        "max_length": 1024,
        "stride": 512,
        "use_cache": False,
        "dtype": "float32",
    },
    "tokenization": {"add_special_tokens": False, "truncation": False, "padding": False},
    "m4b_implemented": False,
    "m4b_reason": "arXiv 2610.10085 の式(6)にある ε と min_scale の実験値が論文に無い。評価用の 1e-12 を流用しない。",
    "versions": {
        "python": sys.version,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "numpy": np.__version__,
    },
    "cuda_device": torch.cuda.get_device_name(0),
    "model_revision": RECORD["model"]["revision"],
    "model_path": RECORD["model"]["path"],
    "wikitext_revision": RECORD["wikitext"]["revision"],
    "c4_revision": RECORD["c4"]["revision"],
    "c4_path": RECORD["c4"]["path"],
    "paths": {
        "data": str(EXPERIMENT_DATA),
        "models": str(MODELS_DIR),
        "results": str(RESULTS_DIR),
    },
}
atomic_write_json(RESULTS_DIR / "settings.json", json_ready(SETTINGS))
print("settings.json を書きました。", torch.cuda.get_device_name(0))

..................                                                       [100%]
18 passed in 2.65s


F                                                                        [100%]
================================== FAILURES ===================================
E   AssertionError: Internal_API.mdにoutput_error_ratios_from_gramの定義元ごとの説明がありません。
    assert 0 >= 1
     +  where 0 = <built-in method count of str object at 0x000001D02571EC80>('`output_error_ratios_from_gram`')
     +    where <built-in method count of str object at 0x000001D02571EC80> = '# Internal helper API境界\n\n**Status:** Internal — Core API v1のPublic APIではない\n\n## 目的\n\n`src/nn_compression/`内で複数のPu...ssion/TTLinear]]、[[90_src設計/05_Core_API_v1/tt_mps/ordered_factorizations]]、[[90_src設計/05_Core_API_v1/tensor/unfold]]\n'.count
D:\dev\nn-compression-svd-dmrg\tests\test_core_api_v1_docs_sync.py:236: AssertionError: Internal_API.mdにoutput_error_ratios_from_gramの定義元ごとの説明がありません。
=========================== short test summary info

## 4.2 データの準備

- 入力: `RECORD`、tokenizer、WikiText-103 train、WikiText-2 validation、C4 の検証シャード
- このステップで行うこと: 見出しで記事を復元し、seed 0 で並べ、1024 トークン以上の文書の先頭 1024 トークンを 1 本だけ取る。較正 128、感度確認用にその先頭 16、同ドメイン検証 32、別ドメイン検証 32。hash の重複を検査する。
- 行わないこと: WikiText の test、モデルの順伝播、圧縮、判定
- 出力: `data_preparation.json`。`CALIBRATION`、`SENSITIVITY`、`VALIDATION_SAME`、`VALIDATION_OTHER`、`PERPLEXITY_IDS`
- 完了条件: 各分割が本数どおり。較正・同ドメイン検証・別ドメイン検証の hash が互いに重ならない。採用系列の hash は分割の中でも一意。
- 次への接続: 4.3 は各分割の `token_ids` と `document_hash` を使う。perplexity は `PERPLEXITY_IDS` を使う。

In [2]:
def cached_encoder(tokenizer, documents, batch_size=8):
    cache = {}
    total = len(documents)
    started = time.perf_counter()
    for start in range(0, total, batch_size):
        batch = list(documents[start : start + batch_size])
        try:
            encoded = tokenizer(
                batch,
                add_special_tokens=False,
                truncation=False,
                padding=False,
                return_attention_mask=False,
            )
            token_rows = encoded["input_ids"]
        except Exception:
            token_rows = [
                tokenizer.encode(text, add_special_tokens=False, truncation=False) for text in batch
            ]
        for text, token_ids in zip(batch, token_rows):
            if torch.is_tensor(token_ids):
                token_ids = token_ids.tolist()
            cache[text] = token_ids
        done = min(start + batch_size, total)
        if start == 0 or done == total or done % (batch_size * 100) == 0:
            print(f"  tokenized {done}/{total} ({time.perf_counter() - started:.1f}s)", flush=True)

    def encode(text):
        return cache[text]

    return encode


def selection_record(selection, name):
    hashes = [sequence.document_hash for sequence in selection.sequences]
    if len(hashes) != len(set(hashes)):
        raise RuntimeError(f"{name} の採用系列に、同じ hash が2回あります。")
    return {
        "name": name,
        "n_documents": selection.n_documents,
        "n_documents_eligible": selection.n_documents_eligible,
        "n_sequences": selection.n_sequences,
        "n_tokens_total": selection.n_tokens_total,
        "n_tokens_kept": selection.n_tokens_kept,
        "n_tokens_excluded": selection.n_tokens_excluded,
        "n_special_tokens": selection.n_special_tokens,
        "n_padding_tokens": selection.n_padding_tokens,
        "sequence_length": SEQUENCE_LENGTH,
        "min_tokens": MIN_TOKENS,
        "seed": SEED,
        "document_hashes": hashes,
    }


def require_exact_count(selection, name, limit):
    if selection.n_sequences != limit:
        raise RuntimeError(f"{name} は {selection.n_sequences} 本です。{limit} 本必要です。")


TOKENIZER = load_tokenizer(RECORD)
print("tokenizer", TOKENIZER.__class__.__name__, flush=True)

print("WikiText-103 train を読みます。", flush=True)
wt103_train = load_wikitext(RECORD, "wikitext-103-raw-v1", "train")
wt103_lines = list(wt103_train["text"])
wt103_articles = restore_wikitext_articles(wt103_lines)
print(f"WikiText-103 行 {len(wt103_lines)}、記事 {len(wt103_articles)}", flush=True)
del wt103_train, wt103_lines
gc.collect()
encode_wt103 = cached_encoder(TOKENIZER, wt103_articles)
CALIBRATION = select_token_prefix_sequences(
    wt103_articles,
    encode_wt103,
    source="wikitext-103-train",
    min_tokens=MIN_TOKENS,
    length=SEQUENCE_LENGTH,
    limit=CALIBRATION_LIMIT,
    seed=SEED,
)
require_exact_count(CALIBRATION, "較正", CALIBRATION_LIMIT)
SENSITIVITY = CALIBRATION.sequences[:SENSITIVITY_LIMIT]
del encode_wt103, wt103_articles
gc.collect()

print("WikiText-2 validation を読みます。", flush=True)
wt2_validation = load_wikitext(RECORD, "wikitext-2-raw-v1", "validation")
wt2_lines = list(wt2_validation["text"])
PERPLEXITY_TEXT = "\n\n".join(wt2_lines)
wt2_articles = restore_wikitext_articles(wt2_lines)
print(f"WikiText-2 validation 行 {len(wt2_lines)}、記事 {len(wt2_articles)}", flush=True)
encode_wt2 = cached_encoder(TOKENIZER, wt2_articles)
VALIDATION_SAME = select_token_prefix_sequences(
    wt2_articles,
    encode_wt2,
    source="wikitext-2-validation",
    min_tokens=MIN_TOKENS,
    length=SEQUENCE_LENGTH,
    limit=VALIDATION_LIMIT,
    seed=SEED,
)
require_exact_count(VALIDATION_SAME, "同ドメイン検証", VALIDATION_LIMIT)
del encode_wt2, wt2_articles
gc.collect()
PERPLEXITY_IDS = torch.tensor(
    TOKENIZER.encode(PERPLEXITY_TEXT, add_special_tokens=False, truncation=False),
    dtype=torch.long,
)
LOGIT_WINDOW = PERPLEXITY_IDS[:SEQUENCE_LENGTH].clone()
if LOGIT_WINDOW.numel() != SEQUENCE_LENGTH:
    raise RuntimeError("perplexity 本文が 1024 トークン未満です。")
del PERPLEXITY_TEXT, wt2_validation, wt2_lines
gc.collect()

print("C4 を読みます。", flush=True)
c4_documents = read_c4_texts(RECORD["c4"]["path"])
print(f"C4 文書 {len(c4_documents)}", flush=True)
encode_c4 = cached_encoder(TOKENIZER, c4_documents)
VALIDATION_OTHER = select_token_prefix_sequences(
    c4_documents,
    encode_c4,
    source="c4-validation-shard0",
    min_tokens=MIN_TOKENS,
    length=SEQUENCE_LENGTH,
    limit=VALIDATION_LIMIT,
    seed=SEED,
)
require_exact_count(VALIDATION_OTHER, "別ドメイン検証", VALIDATION_LIMIT)
del encode_c4, c4_documents
gc.collect()

calibration_record = selection_record(CALIBRATION, "calibration")
same_record = selection_record(VALIDATION_SAME, "validation_same")
other_record = selection_record(VALIDATION_OTHER, "validation_other")
overlaps = hash_overlaps(
    {
        "calibration": calibration_record["document_hashes"],
        "validation_same": same_record["document_hashes"],
        "validation_other": other_record["document_hashes"],
    }
)
if overlaps:
    raise RuntimeError(f"較正と検証の hash が重複しています: {list(overlaps)}")

DATA_PREPARATION = {
    "tokenization": SETTINGS["tokenization"],
    "c4_long_document_bias": "1024 トークン以上の文書だけを採るため、C4 は長い文書に偏る。",
    "sensitivity_is_prefix_of_calibration": True,
    "sensitivity_hashes": [sequence.document_hash for sequence in SENSITIVITY],
    "perplexity_n_tokens": int(PERPLEXITY_IDS.numel()),
    "perplexity_text": "WikiText-2 validation の text 列を改行2つで連結。記事復元後の本文ではない。",
    "hash_overlaps": overlaps,
    "splits": {
        "calibration": calibration_record,
        "validation_same": same_record,
        "validation_other": other_record,
    },
}
atomic_write_json(RESULTS_DIR / "data_preparation.json", json_ready(DATA_PREPARATION))
for name, record in DATA_PREPARATION["splits"].items():
    print(
        name,
        "系列",
        record["n_sequences"],
        "文書",
        record["n_documents"],
        "条件を満たした文書",
        record["n_documents_eligible"],
        "採用トークン",
        record["n_tokens_kept"],
        "除外トークン",
        record["n_tokens_excluded"],
        flush=True,
    )
print("hash の重複はありません。perplexity トークン", int(PERPLEXITY_IDS.numel()), flush=True)

tokenizer Qwen2Tokenizer
WikiText-103 train を読みます。


Using the latest cached version of the dataset since Salesforce/wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'wikitext-103-raw-v1' at D:\dev\nn-compression-svd-dmrg\data\hf_cache\datasets\Salesforce___wikitext\wikitext-103-raw-v1\0.0.0\b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sat Nov  7 17:56:01 2026).


WikiText-103 行 1801350、記事 29444
  tokenized 8/29444 (0.0s)
  tokenized 800/29444 (1.8s)
  tokenized 1600/29444 (3.4s)
  tokenized 2400/29444 (5.0s)
  tokenized 3200/29444 (6.9s)
  tokenized 4000/29444 (8.6s)
  tokenized 4800/29444 (10.2s)
  tokenized 5600/29444 (12.2s)
  tokenized 6400/29444 (14.1s)
  tokenized 7200/29444 (16.2s)
  tokenized 8000/29444 (17.9s)
  tokenized 8800/29444 (20.0s)
  tokenized 9600/29444 (21.8s)
  tokenized 10400/29444 (23.7s)
  tokenized 11200/29444 (25.5s)
  tokenized 12000/29444 (27.5s)
  tokenized 12800/29444 (29.4s)
  tokenized 13600/29444 (31.3s)
  tokenized 14400/29444 (33.2s)
  tokenized 15200/29444 (35.1s)
  tokenized 16000/29444 (36.8s)
  tokenized 16800/29444 (38.6s)
  tokenized 17600/29444 (40.3s)
  tokenized 18400/29444 (42.3s)
  tokenized 19200/29444 (44.2s)
  tokenized 20000/29444 (46.1s)
  tokenized 20800/29444 (47.9s)
  tokenized 21600/29444 (49.8s)
  tokenized 22400/29444 (51.7s)
  tokenized 23200/29444 (53.7s)
  tokenized 24000/29444 (55.7s)

Using the latest cached version of the dataset since Salesforce/wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at D:\dev\nn-compression-svd-dmrg\data\hf_cache\datasets\Salesforce___wikitext\wikitext-2-raw-v1\0.0.0\b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sat Nov  7 17:56:08 2026).


WikiText-2 validation 行 3760、記事 60
  tokenized 8/60 (0.0s)
  tokenized 60/60 (0.2s)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (262337 > 131072). Running this sequence through the model will result in indexing errors


C4 を読みます。
C4 文書 45576
  tokenized 8/45576 (0.0s)
  tokenized 800/45576 (0.3s)
  tokenized 1600/45576 (0.5s)
  tokenized 2400/45576 (0.8s)
  tokenized 3200/45576 (1.1s)
  tokenized 4000/45576 (1.3s)
  tokenized 4800/45576 (1.6s)
  tokenized 5600/45576 (1.9s)
  tokenized 6400/45576 (2.1s)
  tokenized 7200/45576 (2.3s)
  tokenized 8000/45576 (2.6s)
  tokenized 8800/45576 (3.0s)
  tokenized 9600/45576 (3.3s)
  tokenized 10400/45576 (3.5s)
  tokenized 11200/45576 (3.8s)
  tokenized 12000/45576 (4.0s)
  tokenized 12800/45576 (4.3s)
  tokenized 13600/45576 (4.6s)
  tokenized 14400/45576 (4.9s)
  tokenized 15200/45576 (5.2s)
  tokenized 16000/45576 (5.4s)
  tokenized 16800/45576 (5.7s)
  tokenized 17600/45576 (6.0s)
  tokenized 18400/45576 (6.2s)
  tokenized 19200/45576 (6.5s)
  tokenized 20000/45576 (6.8s)
  tokenized 20800/45576 (7.0s)
  tokenized 21600/45576 (7.3s)
  tokenized 22400/45576 (7.5s)
  tokenized 23200/45576 (7.8s)
  tokenized 24000/45576 (8.1s)
  tokenized 24800/45576 (8.4s)
  t

## 4.3 活性化と Gram 行列

- 入力: 4.2 の系列、bf16 の `up_proj` と `gate_proj`
- このステップで行うこと: 重みを float64 で残し、モデルは float32 で GPU に載せる。pre-hook で `up_proj` の入力を取り、float64 で `data/` に保存する。文書ごとの Gram と合計 G を作り、診断を `metric_spectrum.csv` に書く。
- 行わないこと: 圧縮、perplexity、判定。GPU で float64 の計算はしない。
- 出力: `EXPERIMENT_DATA` の活性化と Gram、`metric_spectrum.csv`、`gram_channels.json`。変数 `W`、`W_GATE`、`G`、`G_LAMBDA`、`N_CALIBRATION_TOKENS`、`UP`
- 完了条件: 活性化が (1024, 1024)、float64、CPU。`G_λ` の最小固有値が正。チャネル RMS がすべて正。
- 次への接続: 4.4 は `W` と `G` を使う。4.6 は `UP` とモデルを使う。

In [3]:
def capture_split(name, selection):
    sequences = [torch.tensor(sequence.token_ids, dtype=torch.long) for sequence in selection.sequences]
    document_ids = [sequence.document_hash for sequence in selection.sequences]
    print(f"{name}: 順伝播 {len(sequences)} 本", flush=True)
    started = time.perf_counter()
    records = collect_sequence_inputs(MODEL, UP, sequences, document_ids, device="cuda")
    print(f"{name}: 順伝播 {time.perf_counter() - started:.1f}s", flush=True)
    activation = torch.stack([record["activation"] for record in records])
    if activation.shape != (len(sequences), SEQUENCE_LENGTH, IN_FEATURES):
        raise RuntimeError(f"{name} の活性化 shape が {tuple(activation.shape)} です。")
    if activation.dtype != torch.float64 or activation.device.type != "cpu":
        raise RuntimeError(f"{name} の活性化が float64 CPU ではありません。")
    atomic_torch_save(
        EXPERIMENT_DATA / f"{name}_activations.pt",
        {"document_id": document_ids, "activation": activation},
    )
    one = time.perf_counter()
    gram_matrix(activation[0])
    one_seconds = time.perf_counter() - one
    print(f"{name}: Gram 1文書 {one_seconds:.2f}s、全体の目安 {one_seconds * activation.shape[0] / 60:.1f} 分", flush=True)
    started = time.perf_counter()
    per_document, total = gram_matrices_from_documents(list(activation.unbind(0)))
    print(f"{name}: Gram 全体 {time.perf_counter() - started:.1f}s", flush=True)
    n_tokens = int(activation.shape[0] * activation.shape[1])
    atomic_torch_save(
        EXPERIMENT_DATA / f"{name}_grams.pt",
        {
            "document_id": document_ids,
            "per_document": per_document,
            "total": total,
            "n_tokens": n_tokens,
        },
    )
    diagnostics = gram_diagnostics(total, lam=LAMBDA, n_tokens=n_tokens)
    diagnostics["split"] = name
    del records, sequences, activation, per_document
    gc.collect()
    return total, n_tokens, diagnostics


print("モデルを読みます。", flush=True)
MODEL, _loaded_tokenizer = load_causal_lm_and_tokenizer(RECORD, dtype=torch.bfloat16, device="cpu")
if len(MODEL.model.layers) != 28:
    raise RuntimeError(f"層数が {len(MODEL.model.layers)} です。28 である必要があります。")
MLP = MODEL.model.layers[LAYER_INDEX].mlp
UP = MLP.up_proj
GATE = MLP.gate_proj
if UP.bias is not None or GATE.bias is not None:
    raise RuntimeError("up_proj または gate_proj に bias があります。")
W_BF16 = UP.weight.detach().to(device="cpu").contiguous()
W = W_BF16.to(dtype=torch.float64).contiguous()
W_GATE = GATE.weight.detach().to(device="cpu", dtype=torch.float64).contiguous()
if W.shape != (OUT_FEATURES, IN_FEATURES) or W_GATE.shape != W.shape:
    raise RuntimeError(f"重み shape が W={tuple(W.shape)} gate={tuple(W_GATE.shape)} です。")
WEIGHT_PATH_EQUAL = torch.equal(W.to(dtype=torch.float32), W_BF16.to(dtype=torch.float32))
if not WEIGHT_PATH_EQUAL:
    raise RuntimeError("bf16 → float64 → float32 が bf16 → float32 と bitwise に一致しません。")
atomic_torch_save(EXPERIMENT_DATA / "weights_float64.pt", {"W": W, "W_gate": W_GATE})

MODEL = MODEL.to(dtype=torch.float32, device="cuda")
MODEL.eval()
UP = MODEL.model.layers[LAYER_INDEX].mlp.up_proj
GATE = MODEL.model.layers[LAYER_INDEX].mlp.gate_proj

G, N_CALIBRATION_TOKENS, calibration_diag = capture_split("calibration", CALIBRATION)
G_SAME, N_SAME_TOKENS, same_diag = capture_split("validation_same", VALIDATION_SAME)
G_OTHER, N_OTHER_TOKENS, other_diag = capture_split("validation_other", VALIDATION_OTHER)
G_LAMBDA = regularize_gram(G, LAMBDA)
if float(torch.linalg.eigvalsh(G_LAMBDA)[0].item()) <= 0.0:
    raise RuntimeError("G_λ の最小固有値が正ではありません。")
RMS = channel_rms(G, N_CALIBRATION_TOKENS)
if float(RMS.min().item()) <= 0.0:
    raise RuntimeError("較正のチャネル RMS に 0 以下があります。M4 は値を置き換えないので止めます。")

SPECTRA = [calibration_diag, same_diag, other_diag]
fieldnames = list(SPECTRA[0].keys())
csv_rows = []
for row in SPECTRA:
    csv_rows.append({key: json_ready(row[key]) for key in fieldnames})
atomic_write_csv(RESULTS_DIR / "metric_spectrum.csv", csv_rows, fieldnames)

median = float(RMS.median().item())
order = torch.argsort(RMS, descending=True)
top_channels = []
for index in order[:10].tolist():
    value = float(RMS[index].item())
    top_channels.append(
        {
            "index": int(index),
            "rms": value,
            "rms_over_median": None if median == 0.0 else value / median,
        }
    )
q99 = float(torch.quantile(RMS, 0.99).item())
atomic_write_json(
    RESULTS_DIR / "gram_channels.json",
    json_ready(
        {
            "n_tokens": N_CALIBRATION_TOKENS,
            "rms_min": float(RMS.min().item()),
            "rms_median": median,
            "rms_max": float(RMS.max().item()),
            "n_channels_above_q99": int(torch.count_nonzero(RMS > q99).item()),
            "top10": top_channels,
            "calibration": calibration_diag,
        }
    ),
)
print("較正 G の条件数", calibration_diag["G_condition_number"], flush=True)
print("較正 G の最小固有値", calibration_diag["G_eigenvalue_min"], flush=True)
print("較正 G の effective rank", calibration_diag["G_effective_rank"], flush=True)
print("G_λ の条件数", calibration_diag["G_lambda_condition_number"], flush=True)
print("正則化 / 最小固有値", calibration_diag["regularization_to_min_eigenvalue"], flush=True)
print("RMS 最大 / 中央値", top_channels[0]["rms_over_median"], flush=True)
del G_SAME, G_OTHER
gc.collect()

モデルを読みます。


W1107 20:39:23.936000 42548 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

calibration: 順伝播 128 本
calibration: 順伝播 36.6s
calibration: Gram 1文書 0.01s、全体の目安 0.0 分
calibration: Gram 全体 1.2s
validation_same: 順伝播 32 本
validation_same: 順伝播 8.1s
validation_same: Gram 1文書 0.01s、全体の目安 0.0 分
validation_same: Gram 全体 0.3s
validation_other: 順伝播 32 本
validation_other: 順伝播 7.9s
validation_other: Gram 1文書 0.01s、全体の目安 0.0 分
validation_other: Gram 全体 0.3s
較正 G の条件数 306084136240460.9
較正 G の最小固有値 1.836509024141663e-07
較正 G の effective rank 59.95792817232501
G_λ の条件数 405861302.0920498
正則化 / 最小固有値 754158.4461244743
RMS 最大 / 中央値 26.804925599794462


33

## 4.4 方式

- 入力: `W`、`G`、`N_CALIBRATION_TOKENS`、予算
- このステップで行うこと: T1（4コア）と T2（3コア）を偏り最小で選ぶ。予算ごとの一様 rank と、M3・M4 の実際のパラメータ数を記録する。M1 から M5 の関数を定義する。
- 行わないこと: M5 の掃引、16系列での感度確認の実行、誤差の表、判定。M4b は実装しない。
- 出力: `tensorization.json`、`models/` の M3・M4 コア。`TENSORIZATIONS`、`PARAMETER_COUNTS`、`M3_CORES`
- 完了条件: 選んだ rank の式のパラメータ数が予算以下。コアの要素数が、その rank 列の式と一致する。
- 次への接続: 4.6 は実重みの切り詰めなし TT に mode を使う。4.7 は T2・25% の M3 コアを使う。

In [4]:
def dense_m1(weight, rank):
    return truncated_svd_weight(weight, rank)


def dense_m2(weight, gram_lambda, rank):
    return whitened_truncated_svd(weight, gram_lambda, rank).weight


def fit_m3(weight, out_modes, in_modes, rank_cap):
    return dense_to_tt_matrix_cores(weight, out_modes, in_modes, max_rank=rank_cap)


def dense_from_cores(cores, out_modes, in_modes):
    return tt_matrix_to_dense(cores, out_modes, in_modes)


def fit_m4(weight, gram, n_tokens, out_modes, in_modes, rank_cap):
    return fit_diagonal_metric_tt(weight, gram, n_tokens, out_modes, in_modes, rank_cap)


def dense_m4(fitted, out_modes, in_modes):
    scaled = tt_matrix_to_dense(fitted.scaled_cores, out_modes, in_modes)
    return scale_matrix_columns(scaled, fitted.input_scale_inv)


def initial_m5_from_m3(weight, out_modes, in_modes, rank_cap):
    return fit_m3(weight, out_modes, in_modes, rank_cap)


def initial_m5_from_m4(weight, gram, n_tokens, out_modes, in_modes, rank_cap, target_ranks):
    # M4 の密行列を TT-SVD で作り直したコアを、M3 の rank まで広げる。
    fitted = fit_m4(weight, gram, n_tokens, out_modes, in_modes, rank_cap)
    return prepare_m5_from_diagonal_tt(fitted, out_modes, in_modes, target_ranks)


def run_m5(cores, weight, gram_lambda, out_modes, in_modes):
    if not RUN_FULL_EXPERIMENT:
        raise RuntimeError("M5 の本実行は、事前計測を確認する前には走らせない。")
    return run_metric_tt_als(
        cores,
        weight,
        gram_lambda,
        out_modes,
        in_modes,
        relative_tolerance=CG_RELATIVE_TOLERANCE,
        max_cg_iterations=CG_MAX_ITERATIONS,
        sweep_tolerance=SWEEP_RELATIVE_DECREASE,
        max_sweeps=SWEEP_MAX,
    )


def adopt_lower_calibration_q(candidates):
    """候補は (名前, Q_λ, コア, 履歴)。名前は M3 または M4_retruncated。
    較正の Q_λ が小さい方を採る。同点は M3。
    """
    ordered = sorted(candidates, key=lambda item: (item[1], 0 if item[0] == "M3" else 1))
    name, q_value, cores, history = ordered[0]
    return {"initial": name, "q_lambda": q_value, "cores": cores, "history": history}


TENSORIZATIONS = {}
PARAMETER_COUNTS = []
M3_CORES = {}
M4_FITS = {}
DENSE_MACS = linear_macs(UP)

for n_cores, label in ((4, "T1"), (3, "T2")):
    chosen = select_balanced_tt_matrix_modes(OUT_FEATURES, IN_FEATURES, n_cores)
    out_modes = tuple(chosen["out_modes"])
    in_modes = tuple(chosen["in_modes"])
    budgets = {}
    for budget_name, budget in BUDGETS.items():
        for extra, method in ((0, "M3"), (IN_FEATURES, "M4")):
            selected = select_uniform_tt_rank(out_modes, in_modes, budget, extra_parameters=extra)
            ranks = tuple(selected["ranks"])
            if ranks != tuple(uniform_tt_matrix_ranks(out_modes, in_modes, selected["rank_cap"])):
                raise RuntimeError(f"{label} {budget_name} {method} の rank 列が頭打ち規則と違います。")
            if selected["total_parameters"] > budget:
                raise RuntimeError(f"{label} {budget_name} {method} が予算を超えています。")
            budgets.setdefault(budget_name, {})
            budgets[budget_name][method] = {
                "rank_cap": selected["rank_cap"],
                "ranks": list(ranks),
                "tt_parameters": selected["tt_parameters"],
                "extra_parameters": selected["extra_parameters"],
                "total_parameters": selected["total_parameters"],
            }
        if budgets[budget_name]["M3"]["rank_cap"] != budgets[budget_name]["M4"]["rank_cap"]:
            raise RuntimeError(f"{label} {budget_name} で M3 と M4 の rank 上限が違います。")
    TENSORIZATIONS[label] = {
        "n_cores": n_cores,
        "out_modes": list(out_modes),
        "in_modes": list(in_modes),
        "imbalance": chosen["imbalance"],
        "budgets": budgets,
    }
    print(label, out_modes, in_modes, "imbalance", chosen["imbalance"], flush=True)

for label, spec in TENSORIZATIONS.items():
    out_modes = tuple(spec["out_modes"])
    in_modes = tuple(spec["in_modes"])
    for budget_name, budget in BUDGETS.items():
        plan = spec["budgets"][budget_name]
        rank_cap = plan["M3"]["rank_cap"]
        print(f"TT-SVD {label} {budget_name} cap {rank_cap}", flush=True)
        cores = fit_m3(W, out_modes, in_modes, rank_cap)
        actual_ranks = tuple(tt_matrix_ranks(cores))
        actual_tt = tt_parameter_count(cores)
        formula = tt_matrix_num_parameters_from_ranks(out_modes, in_modes, actual_ranks)
        if actual_tt != formula:
            raise RuntimeError(f"M3 {label} {budget_name} の要素数 {actual_tt} が式 {formula} と違います。")
        M3_CORES[(label, budget_name)] = cores
        atomic_torch_save(
            MODELS_DIR / f"m3_{label}_{budget_name}.pt",
            {"cores": cores, "out_modes": list(out_modes), "in_modes": list(in_modes), "ranks": list(actual_ranks)},
        )
        site_macs = tt_linear_direct_contraction_macs(out_modes, in_modes, actual_ranks)
        PARAMETER_COUNTS.append(
            {
                "tensorization": label,
                "budget": budget_name,
                "method": "M3",
                "rank_cap": rank_cap,
                "formula_ranks": plan["M3"]["ranks"],
                "formula_parameters": plan["M3"]["tt_parameters"],
                "actual_ranks": list(actual_ranks),
                "actual_parameters": actual_tt,
                "actual_matches_rank_formula": True,
                "macs": int(sum(site_macs)),
                "dense_macs": DENSE_MACS,
            }
        )
        fitted = fit_m4(W, G, N_CALIBRATION_TOKENS, out_modes, in_modes, rank_cap)
        m4_ranks = tuple(tt_matrix_ranks(fitted.scaled_cores))
        m4_tt = tt_parameter_count(fitted.scaled_cores)
        m4_formula = tt_matrix_num_parameters_from_ranks(out_modes, in_modes, m4_ranks)
        if m4_tt != m4_formula:
            raise RuntimeError(f"M4 {label} {budget_name} の要素数 {m4_tt} が式 {m4_formula} と違います。")
        if m4_tt + IN_FEATURES > budget:
            raise RuntimeError(f"M4 {label} {budget_name} の実パラメータが予算を超えています。")
        M4_FITS[(label, budget_name)] = fitted
        atomic_torch_save(
            MODELS_DIR / f"m4_{label}_{budget_name}.pt",
            {
                "scaled_cores": fitted.scaled_cores,
                "input_scale_inv": fitted.input_scale_inv,
                "absorbed_cores": fitted.absorbed_cores,
                "scales": fitted.scales,
                "max_rank": fitted.max_rank,
                "ranks": list(m4_ranks),
            },
        )
        m4_macs = tt_linear_direct_contraction_macs(out_modes, in_modes, m4_ranks)
        PARAMETER_COUNTS.append(
            {
                "tensorization": label,
                "budget": budget_name,
                "method": "M4",
                "rank_cap": rank_cap,
                "formula_ranks": plan["M4"]["ranks"],
                "formula_parameters": plan["M4"]["total_parameters"],
                "actual_ranks": list(m4_ranks),
                "actual_tt_parameters": m4_tt,
                "scale_parameters": IN_FEATURES,
                "actual_parameters": m4_tt + IN_FEATURES,
                "actual_matches_rank_formula": True,
                "tt_macs": int(sum(m4_macs)),
                "diagonal_macs": IN_FEATURES,
                "dense_macs": DENSE_MACS,
            }
        )

for budget_name, rank in EXPECTED_MATRIX_RANKS.items():
    parameters = matrix_factor_parameter_count(OUT_FEATURES, IN_FEATURES, rank)
    macs = compressed_linear_macs(IN_FEATURES, OUT_FEATURES, rank)
    for method in ("M1", "M2"):
        PARAMETER_COUNTS.append(
            {
                "tensorization": None,
                "budget": budget_name,
                "method": method,
                "rank": rank,
                "actual_parameters": parameters,
                "macs": macs,
                "dense_macs": DENSE_MACS,
                "actual_matches_rank_formula": True,
            }
        )

atomic_write_json(
    RESULTS_DIR / "tensorization.json",
    json_ready({"tensorizations": TENSORIZATIONS, "counts": PARAMETER_COUNTS, "m4b_implemented": False}),
)
for row in PARAMETER_COUNTS:
    print(row["method"], row.get("tensorization"), row["budget"], "params", row["actual_parameters"], "ranks", row.get("actual_ranks", row.get("rank")), flush=True)


T1 (2, 2, 24, 32) (16, 16, 2, 2) imbalance 2.0
T2 (2, 16, 96) (64, 8, 2) imbalance 1.5
TT-SVD T1 50 cap 382
TT-SVD T1 25 cap 190
TT-SVD T2 50 cap 109
TT-SVD T2 25 cap 77
M3 T1 50 params 1569792 ranks [1, 32, 382, 64, 1]
M4 T1 50 params 1570816 ranks [1, 32, 382, 64, 1]
M3 T1 25 params 783360 ranks [1, 32, 190, 64, 1]
M4 T1 25 params 784384 ranks [1, 32, 190, 64, 1]
M3 T2 50 params 1555648 ranks [1, 109, 109, 1]
M4 T2 50 params 1556672 ranks [1, 109, 109, 1]
M3 T2 25 params 783552 ranks [1, 77, 77, 1]
M4 T2 25 params 784576 ranks [1, 77, 77, 1]
M1 None 50 params 1572864 ranks 384
M2 None 50 params 1572864 ranks 384
M1 None 25 params 786432 ranks 192
M2 None 25 params 786432 ranks 192


## 4.5 評価と判定

- 入力: 重み、近似、Gram または活性化、`statistic_kind`、ゲート重み、perplexity のトークン列
- このステップで行うこと: e、E_rel、E_gated、ブートストラップ、1層置換の perplexity、ρ の判定を関数として定義する。
- 行わないこと: 関数を呼ばない。較正・検証の誤差表を作らない。判定を出さない。感度確認の M2・M5 も走らせない。
- 出力: 関数 `metric_bundle`、`difference_intervals`、`layer_perplexity`、`pilot_decision`
- 完了条件: 評価関数を定義するだけで、このセルでは呼ばない。フラグの真偽では止まらない。
- 次への接続: 本実行では、方式ごとの近似行列をこれらの関数へ渡す。

In [5]:
def activation_ratio_parts(weight, approximated, documents):
    numerators = []
    denominators = []
    delta = weight - approximated
    for document in documents:
        numerators.append(squared_output_error_from_activations(delta, document))
        denominators.append(squared_output_error_from_activations(weight, document))
    return torch.stack(numerators), torch.stack(denominators)


def gram_ratio_parts(weight, approximated, document_grams):
    delta = weight - approximated
    numerators = squared_output_errors_from_documents(delta, document_grams)
    denominators = squared_output_errors_from_documents(weight, document_grams)
    return numerators, denominators


def metric_bundle(weight, approximated, statistic, gate_weight, activations, statistic_kind):
    from nn_compression.compression.metric_low_rank import (
        gated_error_parts,
        output_error_ratios_from_activations,
        output_error_ratios_from_gram,
    )

    if statistic_kind == "gram":
        ratios = output_error_ratios_from_gram(weight, approximated, statistic)
    elif statistic_kind == "activations":
        ratios = output_error_ratios_from_activations(weight, approximated, statistic)
    else:
        raise ValueError(f"statistic_kind は gram または activations です: {statistic_kind}")
    gated_numerator, gated_denominator = gated_error_parts(weight, approximated, gate_weight, activations)
    if float(gated_denominator.item()) == 0.0:
        raise ValueError("E_gated の分母が 0 です。")
    e_gated = gated_numerator / gated_denominator
    ratios["E_gated"] = float(e_gated.item())
    ratios["E_gated_rel"] = float(torch.sqrt(e_gated).item())
    return ratios


def calibration_q_lambda(weight, approximated, gram_lambda):
    return float(squared_output_error_from_gram(weight - approximated, gram_lambda).item())


def difference_intervals(parts_by_method, pairs):
    numerators = {name: parts[0].detach().cpu().numpy() for name, parts in parts_by_method.items()}
    denominators = {name: parts[1].detach().cpu().numpy() for name, parts in parts_by_method.items()}
    return bootstrap_ratio_difference_intervals(
        numerators,
        denominators,
        pairs,
        n_resamples=BOOTSTRAP_RESAMPLES,
        seed=SEED,
        confidence=0.95,
    )


def layer_perplexity(model, module, weight, input_ids):
    with temporary_linear_weight(module, weight):
        return sliding_window_perplexity(
            model,
            input_ids,
            max_length=1024,
            stride=512,
            use_cache=False,
        )


def pilot_decision(cells):
    return classify_pilot_outcome(
        cells,
        tensorizations=("T1", "T2"),
        budgets=("50", "25"),
        validation_domains=("validation_same", "validation_other"),
        calibration_domain="calibration",
        low=RHO_LOW,
        high=RHO_HIGH,
    )


print("評価関数と判定関数を定義した。呼び出しはしない。")

評価関数と判定関数を定義した。呼び出しはしない。


## 4.6 検算

- 入力: `W`、T1/T2 の mode、float32 のモデル、`PERPLEXITY_IDS`、`LOGIT_WINDOW`、4.2 の hash、4.4 のパラメータ数
- このステップで行うこと: 実重みの切り詰めなし TT（検算1）、小さい問題の検算2〜4の数値、検算6（CPU の1窓 logits と GPU の perplexity）、検算7、検算8の記録。`checks.json` に pytest の結果も書く。
- 行わないこと: 検算5のブロック CG。M5。判定。1つでも失敗したら、その場で止める。
- 出力: `checks.json`
- 完了条件: 実重みの相対誤差が 1e-10 以下。CPU logits が bitwise に一致。差し替えた perplexity が、同じ重みを2回測った揺れの範囲に入る。
- 次への接続: 4.7 は、ここを通過した `W` と `G_LAMBDA` と T2 の M3 コアを使う。

In [6]:
def relative_error(left, right):
    return float((torch.linalg.vector_norm(left - right) / torch.linalg.vector_norm(right)).item())


CHECK_RESULTS = {
    "weight_bf16_float64_float32_bitwise_equal": WEIGHT_PATH_EQUAL,
    "check_5_block_cg": {
        "status": "not_applicable",
        "reason": "K⊗I のブロック CG は実装していない。HVP を正とする。",
    },
    "check_8_hash_overlaps": DATA_PREPARATION["hash_overlaps"],
}

print("検算1: 切り詰めなし TT-SVD", flush=True)
exact_errors = {}
for label, spec in TENSORIZATIONS.items():
    out_modes = tuple(spec["out_modes"])
    in_modes = tuple(spec["in_modes"])
    started = time.perf_counter()
    exact_cores = dense_to_tt_matrix_cores(W, out_modes, in_modes, max_rank=None)
    restored = tt_matrix_to_dense(exact_cores, out_modes, in_modes)
    error = relative_error(restored, W)
    exact_errors[label] = error
    print(label, "relative", error, f"{time.perf_counter() - started:.1f}s", flush=True)
    if error > 1e-10:
        atomic_write_json(RESULTS_DIR / "checks.json", json_ready({"check_1_exact_tt": exact_errors, "pytest": PYTEST_LOG}))
        raise RuntimeError(f"検算1が失敗しました。{label} の相対誤差 {error}")
    del exact_cores, restored
    gc.collect()
CHECK_RESULTS["check_1_exact_tt_relative_error"] = exact_errors

print("検算2〜4: 小さい問題", flush=True)
generator = torch.Generator().manual_seed(1)
small_weight = torch.randn(6, 5, generator=generator, dtype=torch.float64)
small_activation = torch.randn(30, 5, generator=generator, dtype=torch.float64)
small_gram = regularize_gram(gram_matrix(small_activation), LAMBDA)
whitened = whitened_truncated_svd(small_weight, small_gram, 2)
plain = truncated_svd_weight(small_weight, 2)
plain_q = squared_output_error_from_gram(small_weight - plain, small_gram)
q_gap = float((whitened.q_lambda - whitened.discarded_singular_value_energy).abs().item())
CHECK_RESULTS["check_2"] = {
    "q_lambda_minus_discarded_energy_abs": q_gap,
    "q_lambda": float(whitened.q_lambda.item()),
    "discarded_energy": float(whitened.discarded_singular_value_energy.item()),
    "plain_q": float(plain_q.item()),
}
if q_gap > 1e-8 or float(whitened.q_lambda.item()) > float(plain_q.item()) + 1e-8:
    raise RuntimeError(f"検算2が失敗しました: {CHECK_RESULTS['check_2']}")

def explicit_hessian(cores, site, gram, out_modes, in_modes):
    core = cores[site]
    dimension = core.numel()
    eye = torch.eye(dimension, dtype=torch.float64)
    rows = math.prod(out_modes) * math.prod(in_modes)
    mapping = torch.empty(rows, dimension, dtype=torch.float64)
    for basis in range(dimension):
        direction = eye[basis].reshape(core.shape)
        replaced = list(cores)
        replaced[site] = direction
        dense = tt_matrix_to_dense(replaced, out_modes, in_modes)
        mapping[:, basis] = dense.transpose(0, 1).contiguous().reshape(-1)
    kronecker = torch.kron(gram, torch.eye(math.prod(out_modes), dtype=torch.float64))
    return mapping.transpose(0, 1) @ kronecker @ mapping

generator = torch.Generator().manual_seed(2)
small_out = (2, 2)
small_in = (2, 2)
small_cores = []
small_ranks = (1, 2, 1)
for site, (out_mode, in_mode) in enumerate(zip(small_out, small_in)):
    small_cores.append(
        torch.randn(small_ranks[site], out_mode, in_mode, small_ranks[site + 1], generator=generator, dtype=torch.float64)
    )
small_cores = canonicalize_tt_matrix_cores(small_cores, small_out, small_in, 0)
small_activation = torch.randn(20, 4, generator=generator, dtype=torch.float64)
small_gram = regularize_gram(gram_matrix(small_activation), LAMBDA)
hessian = explicit_hessian(small_cores, 0, small_gram, small_out, small_in)
direction = torch.randn(small_cores[0].shape, generator=generator, dtype=torch.float64)
applied = metric_tt_hvp(small_cores, 0, direction, small_gram, small_out, small_in).reshape(-1)
reference = hessian @ direction.reshape(-1)
hvp_abs = float((applied - reference).abs().max().item())
diagonal = metric_tt_jacobi_diagonal(small_cores, 0, small_gram, small_out, small_in).reshape(-1)
jacobi_abs = float((diagonal - torch.diagonal(hessian)).abs().max().item())
CHECK_RESULTS["check_3"] = {"hvp_max_abs": hvp_abs, "jacobi_max_abs": jacobi_abs}
if hvp_abs > 1e-8 or jacobi_abs > 1e-8:
    raise RuntimeError(f"検算3が失敗しました: {CHECK_RESULTS['check_3']}")

generator = torch.Generator().manual_seed(3)
center_cores = []
for site, (out_mode, in_mode) in enumerate(zip(small_out, small_in)):
    center_cores.append(
        torch.randn(small_ranks[site], out_mode, in_mode, small_ranks[site + 1], generator=generator, dtype=torch.float64)
    )
center_cores = canonicalize_tt_matrix_cores(center_cores, small_out, small_in, 1)
center_activation = torch.randn(25, 4, generator=generator, dtype=torch.float64)
center_gram = regularize_gram(gram_matrix(center_activation), LAMBDA)
center_hessian = explicit_hessian(center_cores, 1, center_gram, small_out, small_in)
hessian_min = float(torch.linalg.eigvalsh(center_hessian)[0].item())
gram_min = float(torch.linalg.eigvalsh(center_gram)[0].item())
CHECK_RESULTS["check_4"] = {
    "hessian_min": hessian_min,
    "gram_lambda_min": gram_min,
    "difference": hessian_min - gram_min,
}
if hessian_min < gram_min - 1e-8:
    raise RuntimeError(f"検算4が失敗しました: {CHECK_RESULTS['check_4']}")

mismatches = [
    row for row in PARAMETER_COUNTS if row["actual_matches_rank_formula"] is not True
]
CHECK_RESULTS["check_7_real_parameter_counts"] = {
    "n_rows": len(PARAMETER_COUNTS),
    "n_mismatches": len(mismatches),
}
if mismatches:
    raise RuntimeError("検算7が失敗しました。要素数と rank の式が一致しない行があります。")

print("検算6: CPU の 1 窓", flush=True)
window = LOGIT_WINDOW.reshape(1, -1)
MODEL.cpu()
with torch.no_grad():
    logits_direct = MODEL(window).logits
    with temporary_linear_weight(UP, W):
        replaced_cpu = UP.weight.detach().cpu()
        logits_via = MODEL(window).logits
cpu_weight_equal = torch.equal(replaced_cpu, W.to(dtype=torch.float32))
cpu_logits_equal = torch.equal(logits_direct, logits_via)
cpu_max_abs = float((logits_direct - logits_via).abs().max().item())
CHECK_RESULTS["check_6_cpu"] = {
    "weight_equal": cpu_weight_equal,
    "logits_equal": cpu_logits_equal,
    "logits_max_abs": cpu_max_abs,
    "window_tokens": int(window.shape[1]),
}
if not cpu_weight_equal or not cpu_logits_equal:
    atomic_write_json(RESULTS_DIR / "checks.json", json_ready({"checks": CHECK_RESULTS, "pytest": PYTEST_LOG}))
    raise RuntimeError(f"検算6の CPU logits が一致しません。max_abs={cpu_max_abs}")
del logits_direct, logits_via
gc.collect()

print("検算6: GPU perplexity を3回", flush=True)
MODEL.cuda()
torch.cuda.reset_peak_memory_stats()
perplexities = []
for label in ("direct_1", "direct_2"):
    started = time.perf_counter()
    result = sliding_window_perplexity(MODEL, PERPLEXITY_IDS, max_length=1024, stride=512, use_cache=False)
    result = dict(result)
    result["label"] = label
    result["seconds"] = time.perf_counter() - started
    perplexities.append(result)
    print(label, result["perplexity"], result["n_tokens"], result["n_windows"], f"{result['seconds']:.1f}s", flush=True)
started = time.perf_counter()
with temporary_linear_weight(UP, W):
    replaced = sliding_window_perplexity(MODEL, PERPLEXITY_IDS, max_length=1024, stride=512, use_cache=False)
replaced = dict(replaced)
replaced["label"] = "bf16_float64_float32"
replaced["seconds"] = time.perf_counter() - started
print(replaced["label"], replaced["perplexity"], f"{replaced['seconds']:.1f}s", flush=True)
if len({item["n_tokens"] for item in perplexities + [replaced]}) != 1:
    raise RuntimeError("perplexity の評価トークン数が揃っていません。")
noise = abs(perplexities[0]["perplexity"] - perplexities[1]["perplexity"])
delta = abs(replaced["perplexity"] - perplexities[0]["perplexity"])
nll_noise = abs(perplexities[0]["mean_nll"] - perplexities[1]["mean_nll"])
nll_delta = abs(replaced["mean_nll"] - perplexities[0]["mean_nll"])
perplexity_limit = max(10.0 * noise, 1e-8)
nll_limit = max(10.0 * nll_noise, 1e-12)
CHECK_RESULTS["check_6_gpu_perplexity"] = {
    "runs": perplexities + [replaced],
    "perplexity_noise": noise,
    "perplexity_delta": delta,
    "perplexity_limit": perplexity_limit,
    "mean_nll_noise": nll_noise,
    "mean_nll_delta": nll_delta,
    "mean_nll_limit": nll_limit,
    "cuda_max_allocated": int(torch.cuda.max_memory_allocated()),
    "agreement_rule": "差し替えと1回目の差が、1回目と2回目の差の10倍または絶対の下限以下。",
}
if delta > perplexity_limit or nll_delta > nll_limit:
    atomic_write_json(RESULTS_DIR / "checks.json", json_ready({"checks": CHECK_RESULTS, "pytest": PYTEST_LOG}))
    raise RuntimeError(
        f"検算6の GPU perplexity が揺れの範囲を超えています。delta={delta} limit={perplexity_limit}"
    )

CHECK_RESULTS["pytest"] = PYTEST_LOG
atomic_write_json(RESULTS_DIR / "checks.json", json_ready({"checks": CHECK_RESULTS}))
print("checks.json を書きました。", flush=True)

検算1: 切り詰めなし TT-SVD
T1 relative 7.26695432841186e-15 1.1s
T2 relative 4.4924066311278595e-15 0.2s
検算2〜4: 小さい問題
検算6: CPU の 1 窓
検算6: GPU perplexity を3回
direct_1 12.681851366393587 262336 512 113.0s
direct_2 12.681851366393587 262336 512 110.8s
bf16_float64_float32 12.681851366393587 110.8s
checks.json を書きました。


## 4.7 計算時間の事前計測

- 入力: T2、予算 25%、M3 のコア、`W`、`G_LAMBDA`
- このステップで行うこと: コアを site 1 の正準形にし、その1コアだけを CG 10 反復で更新する。HVP 1回、CG 1反復あたり、ピークメモリを測り、M5 を CG 500・掃引 50 まで回した場合の時間を見積もる。
- 行わないこと: 掃引の本実行、他の予算・tensorization・初期値、判定。フラグが True のときは計測自体をしない。
- 出力: `timing.json`
- 完了条件: 計測した更新が1コアだけで、CG の上限が 10。見積もりを書いたらここで止める。
- 次への接続: 見積もりを確認してから、`RUN_FULL_EXPERIMENT` を真にして M5 を回す。この Notebook の実行では真にしない。

In [7]:
if RUN_FULL_EXPERIMENT:
    del MODEL, UP, GATE, MLP, TOKENIZER, _loaded_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("RUN_FULL_EXPERIMENT が True のため、事前計測は飛ばして本実行へ進む。")
else:
    def working_set_bytes():
        import ctypes
        from ctypes import wintypes

        class PROCESS_MEMORY_COUNTERS(ctypes.Structure):
            _fields_ = [
                ("cb", wintypes.DWORD),
                ("PageFaultCount", wintypes.DWORD),
                ("PeakWorkingSetSize", ctypes.c_size_t),
                ("WorkingSetSize", ctypes.c_size_t),
                ("QuotaPeakPagedPoolUsage", ctypes.c_size_t),
                ("QuotaPagedPoolUsage", ctypes.c_size_t),
                ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t),
                ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                ("PagefileUsage", ctypes.c_size_t),
                ("PeakPagefileUsage", ctypes.c_size_t),
            ]

        counters = PROCESS_MEMORY_COUNTERS()
        counters.cb = ctypes.sizeof(PROCESS_MEMORY_COUNTERS)
        kernel32 = ctypes.WinDLL("kernel32", use_last_error=True)
        psapi = ctypes.WinDLL("psapi", use_last_error=True)
        kernel32.GetCurrentProcess.restype = ctypes.c_void_p
        psapi.GetProcessMemoryInfo.argtypes = [ctypes.c_void_p, ctypes.POINTER(PROCESS_MEMORY_COUNTERS), wintypes.DWORD]
        psapi.GetProcessMemoryInfo.restype = wintypes.BOOL
        ok = psapi.GetProcessMemoryInfo(kernel32.GetCurrentProcess(), ctypes.byref(counters), counters.cb)
        if not ok:
            raise OSError("GetProcessMemoryInfo に失敗しました。")
        return int(counters.WorkingSetSize)


    class PeakSampler:
        def __init__(self):
            import threading

            self.peak = working_set_bytes()
            self._stop = threading.Event()
            self._thread = threading.Thread(target=self._run, daemon=True)

        def _run(self):
            while not self._stop.is_set():
                self.peak = max(self.peak, working_set_bytes())
                self._stop.wait(0.05)

        def __enter__(self):
            self._thread.start()
            return self

        def __exit__(self, exc_type, exc, tb):
            self._stop.set()
            self._thread.join()


    del MODEL, UP, GATE, MLP, TOKENIZER, _loaded_tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    spec = TENSORIZATIONS["T2"]
    out_modes = tuple(spec["out_modes"])
    in_modes = tuple(spec["in_modes"])
    rank_cap = spec["budgets"]["25"]["M3"]["rank_cap"]
    cores = canonicalize_tt_matrix_cores(M3_CORES[("T2", "25")], out_modes, in_modes, 1)
    errors = tt_matrix_canonicality_errors(cores, out_modes, in_modes, 1)
    flat_errors = [float(value.item()) for side in errors.values() for value in side]
    if flat_errors and max(flat_errors) > 1e-6:
        raise RuntimeError(f"正準形の誤差が大きいです: {max(flat_errors)}")
    site = 1
    if len(cores) != 3:
        raise RuntimeError(f"T2 のコア数が {len(cores)} です。")
    print("計測サイト", site, tuple(cores[site].shape), "ranks", tt_matrix_ranks(cores), flush=True)

    left = torch.randn(3072, 1024, dtype=torch.float64)
    right = torch.randn(1024, 1024, dtype=torch.float64)
    _ = left @ right
    gemm_started = time.perf_counter()
    _ = left @ right
    gemm_seconds = time.perf_counter() - gemm_started
    del left, right

    direction = torch.randn(cores[site].shape, dtype=torch.float64)
    metric_tt_hvp(cores, site, direction, G_LAMBDA, out_modes, in_modes)
    hvp_samples = []
    hvp_peak = 0
    for _ in range(3):
        with PeakSampler() as sampler:
            started = time.perf_counter()
            metric_tt_hvp(cores, site, direction, G_LAMBDA, out_modes, in_modes)
            hvp_samples.append(time.perf_counter() - started)
        hvp_peak = max(hvp_peak, sampler.peak)
    hvp_seconds = float(sorted(hvp_samples)[1])

    probe_tolerance = 1e-30
    with PeakSampler() as sampler:
        rss_before = working_set_bytes()
        updated, info = update_tt_matrix_site(
            cores,
            site,
            W,
            G_LAMBDA,
            out_modes,
            in_modes,
            relative_tolerance=probe_tolerance,
            max_iterations=10,
            center=site,
        )
        rss_after = working_set_bytes()
    update_peak = sampler.peak
    del updated

    if info.iterations != 10:
        print(f"CG は 10 回に達する前に {info.iterations} 回で止まりました。", flush=True)
    applies = info.iterations + (1 if info.iterations > 0 else 0)
    overhead = max(0.0, info.cg_seconds - applies * hvp_seconds)
    per_iteration_overhead = 0.0 if info.iterations == 0 else overhead / info.iterations
    worst_cg = 501 * hvp_seconds + 500 * per_iteration_overhead
    worst_site = info.jacobi_seconds + worst_cg + info.accept_seconds
    updates = {"T1": 2 * 4 - 1, "T2": 2 * 3 - 1}
    jobs_per_tensorization = 4
    main_seconds = 0.0
    for label, n_updates in updates.items():
        main_seconds += jobs_per_tensorization * SWEEP_MAX * n_updates * worst_site

    TIMING = {
        "condition": {
            "tensorization": "T2",
            "budget": "25",
            "initial": "M3",
            "site": site,
            "core_shape": list(cores[site].shape),
            "ranks": list(tt_matrix_ranks(cores)),
            "rank_cap": rank_cap,
            "cg_iterations_requested": 10,
            "probe_relative_tolerance": probe_tolerance,
            "probe_tolerance_reason": "10反復まで残差で止めないための計測専用の値。本実行の許容値は 1e-8 のまま。",
            "canonicality_max_error": max(flat_errors) if flat_errors else 0.0,
        },
        "hvp_seconds_median": hvp_seconds,
        "hvp_seconds_samples": hvp_samples,
        "reference_gemm_3072x1024_by_1024x1024_seconds": gemm_seconds,
        "update": {
            "accepted": info.accepted,
            "iterations": info.iterations,
            "final_relative_residual": info.final_relative_residual,
            "converged": info.converged,
            "q_before": info.q_before,
            "q_after": info.q_after,
            "jacobi_seconds": info.jacobi_seconds,
            "cg_seconds": info.cg_seconds,
            "accept_seconds": info.accept_seconds,
            "hessian_applies_in_cg": applies,
            "seconds_per_cg_iteration_including_hvp": None if info.iterations == 0 else info.cg_seconds / info.iterations,
        },
        "memory_bytes": {
            "rss_before_update": rss_before,
            "rss_after_update": rss_after,
            "rss_peak_during_update": update_peak,
            "rss_peak_during_hvp": hvp_peak,
        },
        "extrapolation": {
            "assumption": "測った T2・25%・site 1 の1コアを、すべてのサイトとすべての M5 条件に使う。HVP の密行列積は rank にほぼ依らない。コアの再構成は rank が大きい T1・50% の方が重い可能性がある。",
            "worst_cg_iterations": 500,
            "worst_sweeps": SWEEP_MAX,
            "hvp_applies_per_site_at_500": 501,
            "seconds_per_site_at_500": worst_site,
            "updates_per_sweep": updates,
            "jobs": "T1 と T2 のそれぞれで、予算2つ × 初期値2つ = 4本。合計8本。",
            "main_m5_seconds": main_seconds,
            "main_m5_hours": main_seconds / 3600.0,
            "sensitivity_hours_if_same_eight_jobs": main_seconds / 3600.0,
            "sensitivity_note": "16系列の M2 と M5 の組み合わせは設計が格子を指定していない。M5 を主実験と同じ8本とすると、上の時間をもう一度足す。M2 の SVD はこの見積もりには入れていない。",
        },
    }
    atomic_write_json(RESULTS_DIR / "timing.json", json_ready(TIMING))
    print("HVP median", hvp_seconds, flush=True)
    print("CG seconds", info.cg_seconds, "iterations", info.iterations, flush=True)
    print("Jacobi", info.jacobi_seconds, "accept", info.accept_seconds, flush=True)
    print("worst site seconds", worst_site, flush=True)
    print("main M5 hours", main_seconds / 3600.0, flush=True)
    print("ここで止める。本実行と判定には進まない。", flush=True)

RUN_FULL_EXPERIMENT が True のため、事前計測は飛ばして本実行へ進む。


## 5. 本実行の手順

- 入力: 4.4 のコア、4.5 の関数、保存済みの活性化と Gram
- このステップで行うこと: 主実験、感度確認、判定、動作確認が共有する手順を定義する。
- 行わないこと: M5 を走らせない。ファイルを書かない。
- 出力: 関数 `execute_plan`
- 完了条件: 関数が定義され、`RUN_FULL_EXPERIMENT` は False のまま。
- 次への接続: 5.1 以降が `execute_plan` に条件を渡す。

In [8]:
# 本実行と動作確認の共通手順。RUN_FULL_EXPERIMENT が False でも、このセルは関数を定義するだけ。

import csv

from nn_compression.compression.metric_low_rank import gated_error_parts_per_document
from nn_compression.metrics.pilot_decision import RhoEvaluation, evaluate_rho
from nn_compression.tt_mps.metric_tt import DiagonalMetricTT

SUMMARY_COLUMNS = [
    "row_id",
    "calibration_set",
    "tensorization",
    "budget",
    "method",
    "initial",
    "adopted",
    "n_parameters",
    "macs",
    "q",
    "q_lambda",
    "e_calibration",
    "E_rel_calibration",
    "E_gated_calibration",
    "e_validation_same",
    "E_rel_validation_same",
    "E_gated_validation_same",
    "e_validation_other",
    "E_rel_validation_other",
    "E_gated_validation_other",
    "perplexity",
    "mean_nll",
    "n_tokens",
    "seconds",
    "fit_source",
]
HISTORY_COLUMNS = [
    "calibration_set",
    "tensorization",
    "budget",
    "initial",
    "sweep",
    "q_lambda",
    "relative_decrease",
    "cg_iterations",
    "cg_residuals",
    "n_rejected",
    "seconds",
    "stop_reason",
]
INITIAL_COLUMNS = [
    "calibration_set",
    "tensorization",
    "budget",
    "initial",
    "q_lambda_before_als",
    "q_lambda_m4",
    "q_lambda_minus_m4",
]
BOOTSTRAP_COLUMNS = [
    "calibration_set",
    "tensorization",
    "budget",
    "domain",
    "statistic",
    "contrast",
    "estimate",
    "low",
    "high",
    "n_resamples",
]
RHO_COLUMNS = [
    "calibration_set",
    "tensorization",
    "budget",
    "domain",
    "e_m2",
    "e_m3",
    "e_m5",
    "g",
    "g_ci_low",
    "g_ci_high",
    "rho",
    "rho_status",
    "delta_tt",
    "r_m5",
    "kappa",
    "rho_gated",
    "rho_gated_status",
]
PERPLEXITY_COLUMNS = [
    "row_id",
    "calibration_set",
    "tensorization",
    "budget",
    "method",
    "initial",
    "perplexity",
    "mean_nll",
    "n_tokens",
    "n_windows",
    "seconds",
]
EVALUATION_DOMAINS = ("calibration", "validation_same", "validation_other")
BOOTSTRAP_PAIRS = (("M3", "M2"), ("M3", "M5"), ("M5", "M2"))
SPLIT_CACHE = {}
PREPARED_CORES = {}
M4_REFIT = {}
PILOT_MODEL = None
PILOT_UP = None


def empty_state():
    return {
        "summary": {},
        "history": {},
        "initial": {},
        "perplexity": {},
        "completed_m5": [],
        "completed_metrics": [],
        "completed_perplexity": [],
    }


def state_paths(suffix):
    return {
        "json": RESULTS_DIR / f"pilot_state{suffix}.json",
        "parts": RESULTS_DIR / f"error_parts{suffix}.pt",
        "summary": RESULTS_DIR / f"summary{suffix}.csv",
        "history": RESULTS_DIR / f"m5_history{suffix}.csv",
        "initial": RESULTS_DIR / f"m5_initial_values{suffix}.csv",
        "bootstrap": RESULTS_DIR / f"bootstrap{suffix}.csv",
        "rho": RESULTS_DIR / f"rho{suffix}.csv",
        "perplexity": RESULTS_DIR / f"perplexity{suffix}.csv",
    }


def load_run(suffix):
    paths = state_paths(suffix)
    if paths["json"].is_file():
        state = json.loads(paths["json"].read_text(encoding="utf-8"))
        for key, value in empty_state().items():
            state.setdefault(key, value if not isinstance(value, dict) else {})
    else:
        state = empty_state()
    parts = {}
    if paths["parts"].is_file():
        parts = torch.load(paths["parts"], weights_only=False)
    return state, parts


def row_id(calibration_set, tensorization, budget, method, initial):
    return "|".join([calibration_set, tensorization, budget, method, initial])


def m5_key(calibration_set, tensorization, budget, initial):
    return "|".join([calibration_set, tensorization, budget, initial])


def close_enough(left, right):
    scale = max(1.0, abs(left), abs(right))
    return abs(left - right) <= 1e-5 * scale


def load_split(name):
    if name not in SPLIT_CACHE:
        activations = torch.load(EXPERIMENT_DATA / f"{name}_activations.pt", weights_only=False)
        grams = torch.load(EXPERIMENT_DATA / f"{name}_grams.pt", weights_only=False)
        SPLIT_CACHE[name] = {
            "activation": activations["activation"],
            "per_document": grams["per_document"],
            "total": grams["total"],
            "n_tokens": int(grams["n_tokens"]),
        }
    return SPLIT_CACHE[name]


def calibration_bundle(calibration_set):
    saved = load_split("calibration")
    if calibration_set == "full":
        gram = saved["total"]
        return saved["activation"], saved["per_document"], gram, regularize_gram(gram, LAMBDA), saved["n_tokens"]
    if calibration_set != "sensitivity_16":
        raise ValueError(f"calibration_set が不明です: {calibration_set}")
    activation = saved["activation"][:SENSITIVITY_LIMIT]
    per_document = saved["per_document"][:SENSITIVITY_LIMIT]
    gram = (per_document.sum(dim=0) + per_document.sum(dim=0).transpose(0, 1)) * 0.5
    n_tokens = int(activation.shape[0] * activation.shape[1])
    return activation, per_document, gram, regularize_gram(gram, LAMBDA), n_tokens


def m3_core_list(label, budget):
    key = (label, budget)
    if key in M3_CORES:
        return M3_CORES[key]
    payload = torch.load(MODELS_DIR / f"m3_{label}_{budget}.pt", weights_only=False)
    return payload["cores"]


def m4_fit_for(label, budget, gram, n_tokens, out_modes, in_modes, rank_cap, calibration_set):
    if calibration_set == "full" and (label, budget) in M4_FITS:
        return M4_FITS[(label, budget)], "section_4_4"
    if calibration_set == "full":
        payload = torch.load(MODELS_DIR / f"m4_{label}_{budget}.pt", weights_only=False)
        fitted = DiagonalMetricTT(
            scaled_cores=payload["scaled_cores"],
            input_scale_inv=payload["input_scale_inv"],
            absorbed_cores=payload["absorbed_cores"],
            scales=payload["scales"],
            max_rank=payload["max_rank"],
        )
        return fitted, "section_4_4"
    cache_key = (calibration_set, label, budget)
    if cache_key not in M4_REFIT:
        M4_REFIT[cache_key] = fit_m4(W, gram, n_tokens, out_modes, in_modes, rank_cap)
    return M4_REFIT[cache_key], "this_cell"


def prepared_initial(calibration_set, label, budget, initial, gram, n_tokens):
    cache_key = (calibration_set, label, budget, initial)
    if cache_key in PREPARED_CORES:
        return PREPARED_CORES[cache_key]
    spec = TENSORIZATIONS[label]
    out_modes = tuple(spec["out_modes"])
    in_modes = tuple(spec["in_modes"])
    rank_cap = spec["budgets"][budget]["M3"]["rank_cap"]
    target_ranks = tuple(tt_matrix_ranks(m3_core_list(label, budget)))
    if initial == "M3":
        cores = m3_core_list(label, budget)
        source = "section_4_4"
    elif initial == "M4_retruncated":
        fitted, source = m4_fit_for(
            label, budget, gram, n_tokens, out_modes, in_modes, rank_cap, calibration_set
        )
        cores = prepare_m5_from_diagonal_tt(fitted, out_modes, in_modes, target_ranks)
        source = source if calibration_set == "full" else "this_cell"
    else:
        raise ValueError(f"初期値の名前が不明です: {initial}")
    prepared = canonicalize_tt_matrix_cores(cores, out_modes, in_modes, 0)
    PREPARED_CORES[cache_key] = (prepared, out_modes, in_modes, source)
    return PREPARED_CORES[cache_key]


def core_path(suffix, calibration_set, label, budget, initial):
    return MODELS_DIR / f"m5_{calibration_set}_{label}_{budget}_{initial}{suffix}.pt"


def blank_summary(identifier, calibration_set, tensorization, budget, method, initial):
    row = {name: "" for name in SUMMARY_COLUMNS}
    row.update(
        {
            "row_id": identifier,
            "calibration_set": calibration_set,
            "tensorization": tensorization,
            "budget": budget,
            "method": method,
            "initial": initial,
            "adopted": "",
        }
    )
    return row


def attach_metrics(state, parts, identifier, approximated, calibration_set, gram_lambda):
    if identifier in state["completed_metrics"]:
        return
    activation, per_document, gram, _gram_lambda, _n_tokens = calibration_bundle(calibration_set)
    numerator, denominator = gram_ratio_parts(W, approximated, per_document)
    gated_numerator, gated_denominator = gated_error_parts_per_document(
        W, approximated, W_GATE, list(activation.unbind(0))
    )
    bundle = metric_bundle(
        W,
        approximated,
        gram,
        W_GATE,
        activation.reshape(-1, IN_FEATURES),
        "gram",
    )
    ratio = float((numerator.sum() / denominator.sum()).item())
    gated_ratio = float((gated_numerator.sum() / gated_denominator.sum()).item())
    if not close_enough(ratio, bundle["e"]) or not close_enough(gated_ratio, bundle["E_gated"]):
        raise RuntimeError(f"{identifier} の較正誤差が、文書合計と metric_bundle で一致しません。")
    stored = {
        "calibration": {
            "num": numerator,
            "den": denominator,
            "gnum": gated_numerator,
            "gden": gated_denominator,
        }
    }
    scalars = {
        "q": float(squared_output_error_from_gram(W - approximated, gram).item()),
        "q_lambda": calibration_q_lambda(W, approximated, gram_lambda),
        "e_calibration": ratio,
        "E_rel_calibration": math.sqrt(ratio),
        "E_gated_calibration": gated_ratio,
    }
    for split in ("validation_same", "validation_other"):
        saved = load_split(split)
        split_activation = saved["activation"]
        numerator, denominator = activation_ratio_parts(W, approximated, list(split_activation.unbind(0)))
        gated_numerator, gated_denominator = gated_error_parts_per_document(
            W, approximated, W_GATE, list(split_activation.unbind(0))
        )
        flat = split_activation.reshape(-1, IN_FEATURES)
        bundle = metric_bundle(W, approximated, flat, W_GATE, flat, "activations")
        ratio = float((numerator.sum() / denominator.sum()).item())
        gated_ratio = float((gated_numerator.sum() / gated_denominator.sum()).item())
        if not close_enough(ratio, bundle["e"]) or not close_enough(gated_ratio, bundle["E_gated"]):
            raise RuntimeError(f"{identifier} の {split} 誤差が、文書合計と metric_bundle で一致しません。")
        stored[split] = {
            "num": numerator,
            "den": denominator,
            "gnum": gated_numerator,
            "gden": gated_denominator,
        }
        scalars[f"e_{split}"] = ratio
        scalars[f"E_rel_{split}"] = math.sqrt(ratio)
        scalars[f"E_gated_{split}"] = gated_ratio
    parts[identifier] = stored
    state["summary"][identifier].update(scalars)
    state["completed_metrics"].append(identifier)


def mark_adopted(state):
    groups = {}
    for row in state["summary"].values():
        if row["method"] != "M5":
            row["adopted"] = ""
            continue
        key = (row["calibration_set"], row["tensorization"], row["budget"])
        groups.setdefault(key, []).append(row)
    for rows in groups.values():
        best = min(rows, key=lambda item: (float(item["q_lambda"]), 0 if item["initial"] == "M3" else 1))
        for item in rows:
            item["adopted"] = "yes" if item["initial"] == best["initial"] else "no"


def adopted_initial(state, tensorization, budget):
    found = [
        row["initial"]
        for row in state["summary"].values()
        if row["calibration_set"] == "full"
        and row["tensorization"] == tensorization
        and row["budget"] == budget
        and row["method"] == "M5"
        and row["adopted"] == "yes"
    ]
    if len(found) != 1:
        raise RuntimeError(f"主実験の採用初期値が1つに定まっていません: {tensorization} {budget} {found}")
    return found[0]


def matrix_weight(method, budget, gram_lambda):
    rank = EXPECTED_MATRIX_RANKS[budget]
    if method == "M1":
        return truncated_svd_weight(W, rank)
    if method == "M2":
        return whitened_truncated_svd(W, gram_lambda, rank).weight
    raise ValueError(method)


def tt_dense(cores, out_modes, in_modes):
    return tt_matrix_to_dense(cores, out_modes, in_modes)


def parameter_record(method, budget, cores, out_modes, in_modes, extra):
    if method in {"M1", "M2"}:
        rank = EXPECTED_MATRIX_RANKS[budget]
        return (
            matrix_factor_parameter_count(OUT_FEATURES, IN_FEATURES, rank),
            compressed_linear_macs(IN_FEATURES, OUT_FEATURES, rank),
        )
    ranks = tuple(tt_matrix_ranks(cores))
    parameters = tt_parameter_count(cores) + extra
    macs = int(sum(tt_linear_direct_contraction_macs(out_modes, in_modes, ranks))) + extra
    return parameters, macs


def stop_reason(history, max_sweeps, sweep_tolerance):
    if not history:
        return "no_sweep"
    if history[-1].relative_decrease < sweep_tolerance:
        return "relative_decrease"
    if len(history) >= max_sweeps:
        return "max_sweeps"
    return "stopped"


def ensure_model():
    global PILOT_MODEL, PILOT_UP
    if PILOT_MODEL is None:
        model, _tokenizer = load_causal_lm_and_tokenizer(RECORD, dtype=torch.bfloat16, device="cpu")
        model = model.to(dtype=torch.float32, device="cuda")
        model.eval()
        module = model.model.layers[LAYER_INDEX].mlp.up_proj
        if not torch.equal(module.weight.detach().cpu(), W.to(dtype=torch.float32)):
            raise RuntimeError("再読込した up_proj が、bf16 から float64 を経由した重みと一致しません。")
        PILOT_MODEL = model
        PILOT_UP = module
    return PILOT_MODEL, PILOT_UP


def run_one_perplexity(state, suffix, identifier, meta, weight):
    if identifier in state["completed_perplexity"]:
        return
    model, module = ensure_model()
    before = module.weight.detach().clone()
    started = time.perf_counter()
    if weight is None:
        result = sliding_window_perplexity(
            model, PERPLEXITY_IDS, max_length=1024, stride=512, use_cache=False
        )
    else:
        result = layer_perplexity(model, module, weight, PERPLEXITY_IDS)
    elapsed = time.perf_counter() - started
    if not torch.equal(module.weight.detach(), before):
        raise RuntimeError("perplexity の後に up_proj の重みが戻っていません。")
    state["perplexity"][identifier] = {
        "row_id": identifier,
        "calibration_set": meta["calibration_set"],
        "tensorization": meta["tensorization"],
        "budget": meta["budget"],
        "method": meta["method"],
        "initial": meta["initial"],
        "perplexity": result["perplexity"],
        "mean_nll": result["mean_nll"],
        "n_tokens": result["n_tokens"],
        "n_windows": result["n_windows"],
        "seconds": elapsed,
    }
    state["completed_perplexity"].append(identifier)
    if identifier in state["summary"]:
        state["summary"][identifier]["perplexity"] = result["perplexity"]
        state["summary"][identifier]["mean_nll"] = result["mean_nll"]
        state["summary"][identifier]["n_tokens"] = result["n_tokens"]
    save_run(suffix, state, CURRENT_PARTS)
    print(meta["method"], meta["tensorization"], meta["budget"], result["perplexity"], f"{elapsed:.1f}s", flush=True)


def csv_value(value):
    if value is None:
        return ""
    return value


def write_tables(suffix, state, parts):
    paths = state_paths(suffix)
    summary_rows = [state["summary"][key] for key in sorted(state["summary"])]
    atomic_write_csv(paths["summary"], summary_rows, SUMMARY_COLUMNS)
    history_rows = [state["history"][key] for key in sorted(state["history"])]
    atomic_write_csv(paths["history"], history_rows, HISTORY_COLUMNS)
    initial_rows = [state["initial"][key] for key in sorted(state["initial"])]
    atomic_write_csv(paths["initial"], initial_rows, INITIAL_COLUMNS)
    perplexity_rows = [state["perplexity"][key] for key in sorted(state["perplexity"])]
    atomic_write_csv(paths["perplexity"], perplexity_rows, PERPLEXITY_COLUMNS)
    bootstrap_rows = []
    rho_rows = []
    groups = {
        (row["calibration_set"], row["tensorization"], row["budget"])
        for row in state["summary"].values()
        if row["method"] == "M5" and row["adopted"] == "yes"
    }
    for calibration_set, tensorization, budget in sorted(groups):
        adopted = [
            row["initial"]
            for row in state["summary"].values()
            if row["calibration_set"] == calibration_set
            and row["tensorization"] == tensorization
            and row["budget"] == budget
            and row["method"] == "M5"
            and row["adopted"] == "yes"
        ][0]
        method_ids = {
            "M2": row_id(calibration_set, "matrix", budget, "M2", ""),
            "M3": row_id(calibration_set, tensorization, budget, "M3", ""),
            "M5": row_id(calibration_set, tensorization, budget, "M5", adopted),
        }
        if any(identifier not in parts for identifier in method_ids.values()):
            continue
        for domain in EVALUATION_DOMAINS:
            field = "e_calibration" if domain == "calibration" else f"e_{domain}"
            gated_field = "E_gated_calibration" if domain == "calibration" else f"E_gated_{domain}"
            for statistic, num_name, den_name, value_field in (
                ("e", "num", "den", field),
                ("E_gated", "gnum", "gden", gated_field),
            ):
                bundle = {
                    name: (parts[identifier][domain][num_name], parts[identifier][domain][den_name])
                    for name, identifier in method_ids.items()
                }
                intervals = difference_intervals(bundle, BOOTSTRAP_PAIRS)
                for left, right in BOOTSTRAP_PAIRS:
                    estimate = intervals[(left, right)]
                    bootstrap_rows.append(
                        {
                            "calibration_set": calibration_set,
                            "tensorization": tensorization,
                            "budget": budget,
                            "domain": domain,
                            "statistic": statistic,
                            "contrast": f"{left}-{right}",
                            "estimate": estimate["estimate"],
                            "low": estimate["low"],
                            "high": estimate["high"],
                            "n_resamples": estimate["n_resamples"],
                        }
                    )
                if statistic == "e":
                    point_interval = intervals[("M3", "M2")]
                    evaluation = evaluate_rho(
                        budget=budget,
                        tensorization=tensorization,
                        domain=domain,
                        e_m2=float(state["summary"][method_ids["M2"]][value_field]),
                        e_m3=float(state["summary"][method_ids["M3"]][value_field]),
                        e_m5=float(state["summary"][method_ids["M5"]][value_field]),
                        g_ci_low=point_interval["low"],
                        g_ci_high=point_interval["high"],
                        min_relative_g=MIN_RELATIVE_G,
                    )
                    gated_interval = None
                else:
                    gated_interval = intervals[("M3", "M2")]
                    gated_evaluation = evaluate_rho(
                        budget=budget,
                        tensorization=tensorization,
                        domain=domain,
                        e_m2=float(state["summary"][method_ids["M2"]][value_field]),
                        e_m3=float(state["summary"][method_ids["M3"]][value_field]),
                        e_m5=float(state["summary"][method_ids["M5"]][value_field]),
                        g_ci_low=gated_interval["low"],
                        g_ci_high=gated_interval["high"],
                        min_relative_g=MIN_RELATIVE_G,
                    )
                    rho_rows.append(
                        {
                            "calibration_set": calibration_set,
                            "tensorization": tensorization,
                            "budget": budget,
                            "domain": domain,
                            "e_m2": evaluation.e_m2,
                            "e_m3": evaluation.e_m3,
                            "e_m5": evaluation.e_m5,
                            "g": evaluation.g,
                            "g_ci_low": evaluation.g_ci_low,
                            "g_ci_high": evaluation.g_ci_high,
                            "rho": csv_value(evaluation.rho),
                            "rho_status": evaluation.rho_status,
                            "delta_tt": evaluation.delta_tt,
                            "r_m5": evaluation.r_m5,
                            "kappa": csv_value(evaluation.kappa),
                            "rho_gated": csv_value(gated_evaluation.rho),
                            "rho_gated_status": gated_evaluation.rho_status,
                        }
                    )
    atomic_write_csv(paths["bootstrap"], bootstrap_rows, BOOTSTRAP_COLUMNS)
    atomic_write_csv(paths["rho"], rho_rows, RHO_COLUMNS)


def save_run(suffix, state, parts):
    paths = state_paths(suffix)
    atomic_write_json(paths["json"], json_ready(state))
    atomic_torch_save(paths["parts"], parts)
    write_tables(suffix, state, parts)


def resolve_initials(plan, state, label, budget):
    requested = plan["m5_initials"]
    if requested == "adopted_from_full":
        return (adopted_initial(state, label, budget),)
    return tuple(requested)


def execute_plan(plan):
    global CURRENT_PARTS
    suffix = plan["file_suffix"]
    state, parts = load_run(suffix)
    CURRENT_PARTS = parts
    als_ran = []
    als_skipped = []
    for budget in plan["budgets"]:
        _activation, _documents, gram, gram_lambda, n_tokens = calibration_bundle(plan["calibration_set"])
        for method in ("M1", "M2"):
            if method not in plan["methods"]:
                continue
            identifier = row_id(plan["calibration_set"], "matrix", budget, method, "")
            if identifier not in state["completed_metrics"]:
                print("fit", identifier, flush=True)
                started = time.perf_counter()
                approximated = matrix_weight(method, budget, gram_lambda)
                parameters, macs = parameter_record(method, budget, None, None, None, 0)
                row = blank_summary(identifier, plan["calibration_set"], "matrix", budget, method, "")
                row["n_parameters"] = parameters
                row["macs"] = macs
                row["seconds"] = time.perf_counter() - started
                row["fit_source"] = "this_cell"
                state["summary"][identifier] = row
                attach_metrics(state, parts, identifier, approximated, plan["calibration_set"], gram_lambda)
                save_run(suffix, state, parts)
        for label in plan["tensorizations"]:
            spec = TENSORIZATIONS[label]
            out_modes = tuple(spec["out_modes"])
            in_modes = tuple(spec["in_modes"])
            rank_cap = spec["budgets"][budget]["M3"]["rank_cap"]
            fitted, m4_source = m4_fit_for(
                label,
                budget,
                gram,
                n_tokens,
                out_modes,
                in_modes,
                rank_cap,
                plan["calibration_set"],
            )
            missing_initials = [
                initial
                for initial in plan["record_initials"]
                if "|".join([plan["calibration_set"], label, budget, initial]) not in state["initial"]
            ]
            if missing_initials:
                q_m4 = calibration_q_lambda(W, dense_m4(fitted, out_modes, in_modes), gram_lambda)
            for initial in missing_initials:
                initial_id = "|".join([plan["calibration_set"], label, budget, initial])
                prepared, _out, _inn, _source = prepared_initial(
                    plan["calibration_set"], label, budget, initial, gram, n_tokens
                )
                q_before = float(q_lambda_of_cores(prepared, W, gram_lambda, out_modes, in_modes).item())
                state["initial"][initial_id] = {
                    "calibration_set": plan["calibration_set"],
                    "tensorization": label,
                    "budget": budget,
                    "initial": initial,
                    "q_lambda_before_als": q_before,
                    "q_lambda_m4": q_m4,
                    "q_lambda_minus_m4": q_before - q_m4,
                }
                save_run(suffix, state, parts)
            if "M3" in plan["methods"]:
                identifier = row_id(plan["calibration_set"], label, budget, "M3", "")
                if identifier not in state["completed_metrics"]:
                    print("fit", identifier, flush=True)
                    started = time.perf_counter()
                    cores = m3_core_list(label, budget)
                    approximated = tt_dense(cores, out_modes, in_modes)
                    parameters, macs = parameter_record("M3", budget, cores, out_modes, in_modes, 0)
                    row = blank_summary(identifier, plan["calibration_set"], label, budget, "M3", "")
                    row["n_parameters"] = parameters
                    row["macs"] = macs
                    row["seconds"] = time.perf_counter() - started
                    row["fit_source"] = "section_4_4"
                    state["summary"][identifier] = row
                    attach_metrics(state, parts, identifier, approximated, plan["calibration_set"], gram_lambda)
                    save_run(suffix, state, parts)
            if "M4" in plan["methods"]:
                identifier = row_id(plan["calibration_set"], label, budget, "M4", "")
                if identifier not in state["completed_metrics"]:
                    print("fit", identifier, flush=True)
                    started = time.perf_counter()
                    approximated = dense_m4(fitted, out_modes, in_modes)
                    parameters, macs = parameter_record(
                        "M4", budget, fitted.scaled_cores, out_modes, in_modes, IN_FEATURES
                    )
                    row = blank_summary(identifier, plan["calibration_set"], label, budget, "M4", "")
                    row["n_parameters"] = parameters
                    row["macs"] = macs
                    row["seconds"] = time.perf_counter() - started
                    row["fit_source"] = m4_source
                    state["summary"][identifier] = row
                    attach_metrics(state, parts, identifier, approximated, plan["calibration_set"], gram_lambda)
                    save_run(suffix, state, parts)
            for initial in resolve_initials(plan, state, label, budget):
                key = m5_key(plan["calibration_set"], label, budget, initial)
                identifier = row_id(plan["calibration_set"], label, budget, "M5", initial)
                path = core_path(suffix, plan["calibration_set"], label, budget, initial)
                final_cores = None
                if key in state["completed_m5"]:
                    als_skipped.append(key)
                    print("skip M5", key, flush=True)
                    if identifier not in state["completed_metrics"]:
                        payload = torch.load(path, weights_only=False)
                        final_cores = payload["cores"]
                else:
                    als_ran.append(key)
                    print("ALS", key, flush=True)
                    prepared, out_modes, in_modes, fit_source = prepared_initial(
                        plan["calibration_set"], label, budget, initial, gram, n_tokens
                    )
                    started = time.perf_counter()
                    if plan["use_experiment_stop"]:
                        final_cores, history = run_m5(prepared, W, gram_lambda, out_modes, in_modes)
                    else:
                        final_cores, history = run_metric_tt_als(
                            prepared,
                            W,
                            gram_lambda,
                            out_modes,
                            in_modes,
                            relative_tolerance=plan["relative_tolerance"],
                            max_cg_iterations=plan["max_cg_iterations"],
                            sweep_tolerance=plan["sweep_tolerance"],
                            max_sweeps=plan["max_sweeps"],
                        )
                    reason = stop_reason(history, plan["max_sweeps"], plan["sweep_tolerance"])
                    atomic_torch_save(
                        path,
                        {
                            "cores": final_cores,
                            "out_modes": list(out_modes),
                            "in_modes": list(in_modes),
                            "initial": initial,
                        },
                    )
                    for record in history:
                        history_id = f"{key}|{record.sweep:03d}"
                        state["history"][history_id] = {
                            "calibration_set": plan["calibration_set"],
                            "tensorization": label,
                            "budget": budget,
                            "initial": initial,
                            "sweep": record.sweep,
                            "q_lambda": record.q_lambda,
                            "relative_decrease": record.relative_decrease,
                            "cg_iterations": ";".join(str(item) for item in record.cg_iterations),
                            "cg_residuals": ";".join(f"{item:.6e}" for item in record.cg_residuals),
                            "n_rejected": record.n_rejected,
                            "seconds": record.seconds,
                            "stop_reason": reason,
                        }
                    parameters, macs = parameter_record("M5", budget, final_cores, out_modes, in_modes, 0)
                    row = blank_summary(
                        identifier, plan["calibration_set"], label, budget, "M5", initial
                    )
                    row["n_parameters"] = parameters
                    row["macs"] = macs
                    row["seconds"] = time.perf_counter() - started
                    row["fit_source"] = fit_source
                    state["summary"][identifier] = row
                    state["completed_m5"].append(key)
                    save_run(suffix, state, parts)
                if identifier not in state["completed_metrics"]:
                    if final_cores is None:
                        payload = torch.load(path, weights_only=False)
                        final_cores = payload["cores"]
                    approximated = tt_dense(final_cores, out_modes, in_modes)
                    attach_metrics(state, parts, identifier, approximated, plan["calibration_set"], gram_lambda)
                    matching = [
                        item
                        for item in state["history"].values()
                        if item["calibration_set"] == plan["calibration_set"]
                        and item["tensorization"] == label
                        and item["budget"] == budget
                        and item["initial"] == initial
                    ]
                    if matching:
                        last_q = max(matching, key=lambda item: int(item["sweep"]))["q_lambda"]
                        if not close_enough(state["summary"][identifier]["q_lambda"], last_q):
                            raise RuntimeError(f"{identifier} の Q_λ が掃引履歴と一致しません。")
                    save_run(suffix, state, parts)
    mark_adopted(state)
    if plan["compute_perplexity"]:
        run_one_perplexity(
            state,
            suffix,
            "full_model",
            {
                "calibration_set": "full",
                "tensorization": "",
                "budget": "",
                "method": "full_model",
                "initial": "",
            },
            None,
        )
        for row in list(state["summary"].values()):
            if row["calibration_set"] != plan["calibration_set"]:
                continue
            if row["tensorization"] not in plan["tensorizations"] and row["tensorization"] != "matrix":
                continue
            if row["budget"] not in plan["budgets"]:
                continue
            if row["method"] == "M5" and row["adopted"] != "yes":
                continue
            if row["row_id"] in state["completed_perplexity"]:
                continue
            weight = materialize_row(plan, state, row)
            run_one_perplexity(state, suffix, row["row_id"], row, weight)
    save_run(suffix, state, parts)
    return {"als_ran": als_ran, "als_skipped": als_skipped, "state": state}


def materialize_row(plan, state, row):
    _activation, _documents, gram, gram_lambda, n_tokens = calibration_bundle(row["calibration_set"])
    if row["method"] in {"M1", "M2"}:
        return matrix_weight(row["method"], row["budget"], gram_lambda)
    spec = TENSORIZATIONS[row["tensorization"]]
    out_modes = tuple(spec["out_modes"])
    in_modes = tuple(spec["in_modes"])
    if row["method"] == "M3":
        return tt_dense(m3_core_list(row["tensorization"], row["budget"]), out_modes, in_modes)
    if row["method"] == "M4":
        rank_cap = spec["budgets"][row["budget"]]["M3"]["rank_cap"]
        fitted, _source = m4_fit_for(
            row["tensorization"],
            row["budget"],
            gram,
            n_tokens,
            out_modes,
            in_modes,
            rank_cap,
            row["calibration_set"],
        )
        return dense_m4(fitted, out_modes, in_modes)
    if row["method"] == "M5":
        payload = torch.load(
            core_path(plan["file_suffix"], row["calibration_set"], row["tensorization"], row["budget"], row["initial"]),
            weights_only=False,
        )
        return tt_dense(payload["cores"], out_modes, in_modes)
    raise ValueError(row["method"])


CURRENT_PARTS = {}

## 5.1 主実験

- 入力: `execute_plan`、`RUN_FULL_EXPERIMENT`
- このステップで行うこと: フラグが真のとき、T1・T2、予算 50%・25%、M1〜M4 と M5 の初期値 M3・M4_retruncated を計算する。
- 行わないこと: フラグが偽のときは何も計算しない。λ と停止条件は変えない。
- 出力: `summary.csv`、`m5_history.csv`、`m5_initial_values.csv`、`bootstrap.csv`、`rho.csv`、`perplexity.csv`。コアは `models/`。
- 完了条件: フラグが偽なら、新しい結果ファイルを作らずに終わる。
- 次への接続: 5.2 は、ここで採用された初期値を読む。

In [9]:
if not RUN_FULL_EXPERIMENT:
    print("RUN_FULL_EXPERIMENT が False のため、主実験はしない。")
else:
    execute_plan(
        {
            "calibration_set": "full",
            "tensorizations": ("T1", "T2"),
            "budgets": ("50", "25"),
            "methods": ("M1", "M2", "M3", "M4", "M5"),
            "m5_initials": ("M3", "M4_retruncated"),
            "record_initials": ("M3", "M4_retruncated"),
            "max_sweeps": SWEEP_MAX,
            "max_cg_iterations": CG_MAX_ITERATIONS,
            "relative_tolerance": CG_RELATIVE_TOLERANCE,
            "sweep_tolerance": SWEEP_RELATIVE_DECREASE,
            "file_suffix": "",
            "compute_perplexity": True,
            "use_experiment_stop": True,
        }
    )

fit full|matrix|50|M1|
fit full|matrix|50|M2|
fit full|T1|50|M3|
fit full|T1|50|M4|
ALS full|T1|50|M3
ALS full|T1|50|M4_retruncated
fit full|T2|50|M3|
fit full|T2|50|M4|
ALS full|T2|50|M3
ALS full|T2|50|M4_retruncated
fit full|matrix|25|M1|
fit full|matrix|25|M2|
fit full|T1|25|M3|
fit full|T1|25|M4|
ALS full|T1|25|M3
ALS full|T1|25|M4_retruncated
fit full|T2|25|M3|
fit full|T2|25|M4|
ALS full|T2|25|M3
ALS full|T2|25|M4_retruncated


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

full_model   12.681851366393587 54.8s
M1 matrix 50 13.010836381180228 55.0s
M2 matrix 50 12.759716590789212 54.8s
M3 T1 50 13.140889186364557 54.8s
M4 T1 50 12.916538550759377 54.8s
M5 T1 50 12.788789998297737 54.9s
M3 T2 50 13.185779347595354 54.9s
M4 T2 50 12.993015275837989 54.8s
M5 T2 50 12.835851981029812 54.8s
M1 matrix 25 13.147880865824598 54.7s
M2 matrix 25 12.863134851958389 54.7s
M3 T1 25 13.185088894869304 54.8s
M4 T1 25 13.098577796106614 54.7s
M5 T1 25 12.906033241918626 54.9s
M3 T2 25 13.227871343646054 54.7s
M4 T2 25 13.1941054627915 54.7s
M5 T2 25 12.982118821623857 54.8s


## 5.2 較正量の感度確認

- 入力: 主実験で採用された初期値、較正系列の先頭 16
- このステップで行うこと: フラグが真のとき、T2 の予算 50%・25% で G を 16 系列から作り直し、M2 と、採用された初期値の M5 を計算する。ρ の分母に使う M3 は、主実験と同じ TT-SVD をこの 16 系列で測り直す。
- 行わないこと: フラグが偽のときは何もしない。判定には使わない。perplexity は測らない。
- 出力: 5.1 と同じ CSV に、`calibration_set=sensitivity_16` の行を足す。
- 完了条件: フラグが偽ならファイルを増やさない。
- 次への接続: 5.3 は `calibration_set=full` の行だけを判定に使う。

In [10]:
if not RUN_FULL_EXPERIMENT:
    print("RUN_FULL_EXPERIMENT が False のため、感度確認はしない。")
else:
    execute_plan(
        {
            "calibration_set": "sensitivity_16",
            "tensorizations": ("T2",),
            "budgets": ("50", "25"),
            "methods": ("M2", "M3", "M5"),
            "m5_initials": "adopted_from_full",
            "record_initials": ("M3", "M4_retruncated"),
            "max_sweeps": SWEEP_MAX,
            "max_cg_iterations": CG_MAX_ITERATIONS,
            "relative_tolerance": CG_RELATIVE_TOLERANCE,
            "sweep_tolerance": SWEEP_RELATIVE_DECREASE,
            "file_suffix": "",
            "compute_perplexity": False,
            "use_experiment_stop": True,
        }
    )

fit sensitivity_16|matrix|50|M2|
fit sensitivity_16|T2|50|M3|
ALS sensitivity_16|T2|50|M3
fit sensitivity_16|matrix|25|M2|
fit sensitivity_16|T2|25|M3|
ALS sensitivity_16|T2|25|M3


## 5.3 判定

- 入力: `rho.csv` の `calibration_set=full` の行
- このステップで行うこと: フラグが真のとき、`pilot_decision` で判定し、使ったセルと一緒に `decision.json` へ書く。
- 行わないこと: フラグが偽のときは判定しない。感度確認の行は使わない。
- 出力: `decision.json`
- 完了条件: フラグが偽なら `decision.json` を作らない。
- 次への接続: 動作確認は別ファイル名で、この判定を呼ばない。

In [11]:
if not RUN_FULL_EXPERIMENT:
    print("RUN_FULL_EXPERIMENT が False のため、判定は出さない。")
else:
    rho_path = RESULTS_DIR / "rho.csv"
    with rho_path.open(encoding="utf-8", newline="") as handle:
        rho_rows = list(csv.DictReader(handle))
    cells = []
    recorded = []
    for row in rho_rows:
        if row["calibration_set"] != "full":
            continue
        rho = None if row["rho"] == "" else float(row["rho"])
        kappa = None if row["kappa"] == "" else float(row["kappa"])
        cells.append(
            RhoEvaluation(
                budget=row["budget"],
                tensorization=row["tensorization"],
                domain=row["domain"],
                e_m2=float(row["e_m2"]),
                e_m3=float(row["e_m3"]),
                e_m5=float(row["e_m5"]),
                g_ci_low=float(row["g_ci_low"]),
                g_ci_high=float(row["g_ci_high"]),
                g=float(row["g"]),
                rho=rho,
                rho_status=row["rho_status"],
                delta_tt=float(row["delta_tt"]),
                r_m5=float(row["r_m5"]),
                kappa=kappa,
            )
        )
        recorded.append(
            {
                "budget": row["budget"],
                "tensorization": row["tensorization"],
                "domain": row["domain"],
                "e_m2": float(row["e_m2"]),
                "e_m3": float(row["e_m3"]),
                "e_m5": float(row["e_m5"]),
                "g": float(row["g"]),
                "g_ci_low": float(row["g_ci_low"]),
                "g_ci_high": float(row["g_ci_high"]),
                "rho": rho,
                "rho_status": row["rho_status"],
                "delta_tt": float(row["delta_tt"]),
                "r_m5": float(row["r_m5"]),
                "kappa": kappa,
            }
        )
    outcome = pilot_decision(cells)
    atomic_write_json(
        RESULTS_DIR / "decision.json",
        json_ready({"outcome": outcome, "cells": recorded}),
    )
    print(outcome)

本実験へ進む


## 5.4 動作確認

- 入力: 5. の `execute_plan`
- このステップで行うこと: T2、予算 25%、M5 の初期値は M3 だけ、掃引 1 回、CG 10 反復で最後まで書く。終わった状態でもう一度実行し、その M5 を飛ばす。続けて 16 系列の感度確認を 1 本だけ走らせる。
- 行わないこと: フラグが True のときは、このセルでは何も計測しない。False のときは本実行の停止条件では走らせない。`decision.json` は出さない。ファイル名には `_smoke` を付ける。
- 出力: `summary_smoke.csv` ほか `_smoke` の付いた CSV と、`models/` の `*_smoke.pt`
- 完了条件: フラグが False のときは、列名が定義どおりで、2 回目は M5 を 1 本も走らせず、重みの差し替えが元に戻る。判定ファイルは無い。フラグが True のときはファイルを消さず、本実行の結果も触らない。
- 次への接続: 結果を見てから、5.1 のフラグを真にして本実行する。

In [12]:
if RUN_FULL_EXPERIMENT:
    print("RUN_FULL_EXPERIMENT が True のため、動作確認は飛ばす。")
else:
    def remove_smoke_outputs():
        for path in RESULTS_DIR.glob("*_smoke.csv"):
            path.unlink()
        for name in ("pilot_state_smoke.json", "error_parts_smoke.pt"):
            (RESULTS_DIR / name).unlink(missing_ok=True)
        for path in MODELS_DIR.glob("*_smoke.pt"):
            path.unlink()


    def assert_header(path, columns):
        header = path.read_text(encoding="utf-8").splitlines()[0]
        if header.split(",") != list(columns):
            raise RuntimeError(f"{path.name} の列が違います: {header}")


    SMOKE_MAIN = {
        "calibration_set": "full",
        "tensorizations": ("T2",),
        "budgets": ("25",),
        "methods": ("M1", "M2", "M3", "M4", "M5"),
        "m5_initials": ("M3",),
        "record_initials": ("M3", "M4_retruncated"),
        "max_sweeps": 1,
        "max_cg_iterations": 10,
        "relative_tolerance": CG_RELATIVE_TOLERANCE,
        "sweep_tolerance": SWEEP_RELATIVE_DECREASE,
        "file_suffix": "_smoke",
        "compute_perplexity": True,
        "use_experiment_stop": False,
    }
    SMOKE_SENSITIVITY = {
        "calibration_set": "sensitivity_16",
        "tensorizations": ("T2",),
        "budgets": ("25",),
        "methods": ("M2", "M3", "M5"),
        "m5_initials": "adopted_from_full",
        "record_initials": ("M3", "M4_retruncated"),
        "max_sweeps": 1,
        "max_cg_iterations": 10,
        "relative_tolerance": CG_RELATIVE_TOLERANCE,
        "sweep_tolerance": SWEEP_RELATIVE_DECREASE,
        "file_suffix": "_smoke",
        "compute_perplexity": False,
        "use_experiment_stop": False,
    }
    smoke_started = time.perf_counter()
    remove_smoke_outputs()
    first = execute_plan(SMOKE_MAIN)
    summary_text = (RESULTS_DIR / "summary_smoke.csv").read_text(encoding="utf-8")
    second = execute_plan(SMOKE_MAIN)
    if (RESULTS_DIR / "summary_smoke.csv").read_text(encoding="utf-8") != summary_text:
        raise RuntimeError("再開の再実行で summary_smoke.csv の内容が変わりました。")
    if first["als_ran"] != ["full|T2|25|M3"]:
        raise RuntimeError(f"動作確認の M5 が想定と違います: {first['als_ran']}")
    if second["als_ran"] or second["als_skipped"] != ["full|T2|25|M3"]:
        raise RuntimeError(
            f"再開で M5 を飛ばせていません: ran={second['als_ran']} skipped={second['als_skipped']}"
        )
    third = execute_plan(SMOKE_SENSITIVITY)
    if third["als_ran"] != ["sensitivity_16|T2|25|M3"]:
        raise RuntimeError(f"感度確認の M5 が想定と違います: {third['als_ran']}")
    if (RESULTS_DIR / "decision.json").exists():
        raise RuntimeError("動作確認が decision.json を作りました。")
    assert_header(RESULTS_DIR / "summary_smoke.csv", SUMMARY_COLUMNS)
    assert_header(RESULTS_DIR / "m5_history_smoke.csv", HISTORY_COLUMNS)
    assert_header(RESULTS_DIR / "m5_initial_values_smoke.csv", INITIAL_COLUMNS)
    assert_header(RESULTS_DIR / "bootstrap_smoke.csv", BOOTSTRAP_COLUMNS)
    assert_header(RESULTS_DIR / "rho_smoke.csv", RHO_COLUMNS)
    assert_header(RESULTS_DIR / "perplexity_smoke.csv", PERPLEXITY_COLUMNS)
    for name in (
        "summary_smoke.csv",
        "m5_history_smoke.csv",
        "m5_initial_values_smoke.csv",
        "bootstrap_smoke.csv",
        "rho_smoke.csv",
        "perplexity_smoke.csv",
    ):
        lines = (RESULTS_DIR / name).read_text(encoding="utf-8").splitlines()
        print(f"--- {name} ({len(lines) - 1} 行) ---")
        print("\n".join(lines[:4]))
    initial = third["state"]["initial"]["full|T2|25|M4_retruncated"]
    print(
        "T2 25% M4_retruncated Q_lambda",
        initial["q_lambda_before_als"],
        "M4 Q_lambda",
        initial["q_lambda_m4"],
        "差",
        initial["q_lambda_minus_m4"],
    )
    print(f"動作確認 {time.perf_counter() - smoke_started:.1f}s")
    print("ここで止める。本実行と判定には進まない。")

RUN_FULL_EXPERIMENT が True のため、動作確認は飛ばす。
